# Planificación adaptativa de entrenamientos de alta carga mediante aprendizaje automático
## Caso de estudio del triatlón de media distancia (113 km) — Trabajo Fin de Máster
**Autor:** Marco Hidalgo Sanguano | **Director:** José Manuel Díaz Sarachaga

Este cuaderno contiene el análisis completo que se describe en la memoria. Sigue las tres fases del
capítulo 3:

| Fase | Contenido | Memoria |
|---|---|---|
| 0. Datos | Carga de las exportaciones de Garmin Connect y de los conjuntos públicos, limpieza, TRIMP y CTL/ATL/TSB | 3.3, 4.2, 5.1 |
| 1. Transferencia | Dos modelos XGBoost: ciclismo a partir de la carrera y carrera a partir del ciclismo | 3.4, 4.4, 5.2 |
| 2. Planificación y explicación | Reglas de tipo de semana (Tabla 7), periodización, zonas, plan y texto explicativo | 3.5, 4.5, 4.6, 5.3 |

La sección 5 conserva el agrupamiento K-means que se probó y se descartó (apartado 6.2); nada de esa
sección se usa en el resto del cuaderno ni en la aplicación.

### Deportistas
En la memoria aparecen con letras. El cuaderno usa las mismas letras en todas sus salidas, de modo que las
tablas coinciden con las de la memoria y no se muestran nombres.

| Deportista | Historial principal | Papel en el estudio |
|---|---|---|
| A (mujer) | Carrera y natación, solo 15 sesiones de ciclismo | Caso de evaluación de la estimación entre disciplinas |
| B | Sobre todo carrera | Contraste |
| C | Las tres disciplinas, el mayor volumen | Contraste |
| D | Sobre todo ciclismo, casi sin frecuencia cardíaca | Contraste |

### Conjuntos públicos (Kaggle)
| Conjunto | Uso |
|---|---|
| Triathlon results (vladislavboyadzhi) | Entrenamiento y validación de los modelos de transferencia |
| Athlete training and recovery tracker (prince7489) | Exploración de variables de recuperación |
| GoldenCheetah OpenData (markliversedge) | Exploración inicial de variables de carga |

### Salidas
Al terminar, la carpeta `salidas/` contiene los modelos que usa la aplicación (`salidas/modelos/`), las
tablas en Parquet y SQLite y las figuras de la memoria (`salidas/figuras/`).

## 0. Instalación y configuración

In [ ]:
# Paquetes que no vienen instalados por defecto en Colab. XGBoost y scikit-learn se fijan
# en las mismas versiones que la aplicación (requirements.txt) para que los modelos
# guardados aquí se carguen allí sin problemas de compatibilidad.
!pip install -q kagglehub "xgboost==2.1.4" "scikit-learn==1.6.1" shap plotly google-generativeai pyarrow
print("Dependencias instaladas")

In [ ]:
import os, glob, json, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold, cross_validate
from sklearn.impute import SimpleImputer

import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

# Letra de cada deportista en la memoria -> nombre de su ZIP de Garmin Connect
PARTICIPANTES = {'A': 'Josh', 'B': 'Marco', 'C': 'Andresin', 'D': 'Patico'}
sexo_por_atleta = {'A': 'F', 'B': 'M', 'C': 'M', 'D': 'M'}   # para el TRIMP y los modelos
SUJETO_PRUEBA = 'A'   # caso de evaluación de la estimación entre disciplinas (memoria, 3.2)
COLORS = {'A': '#E91E63', 'B': '#2196F3', 'C': '#FF5722', 'D': '#4CAF50'}

# Figuras de la memoria: fondo blanco y texto negro
os.makedirs('salidas/figuras', exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.titleweight': 'bold', 'axes.titlesize': 11})
print("Librerías cargadas")

In [ ]:
# API de Gemini para la capa explicativa. Es opcional: si no hay clave
# se usan plantillas de texto fijas y el cuaderno funciona igual.
GEMINI_API_KEY = ""   # pega aquí la clave de https://aistudio.google.com/
USE_LLM = bool(GEMINI_API_KEY)
if USE_LLM:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    llm_model = genai.GenerativeModel('gemini-1.5-flash')
    print("Gemini configurado")
else:
    print("Sin clave de Gemini: se usarán plantillas de texto")

## 1. Carga de los conjuntos públicos

Memoria, apartado 3.2. Los resultados de pruebas 70.3 aportan el volumen necesario para entrenar los
modelos sin depender de los cuatro deportistas. Los otros dos conjuntos solo se exploran.

In [ ]:
import kagglehub

# 1.1 Resultados de triatlón: tiempos parciales reales de competición
path_tri = kagglehub.dataset_download("vladislavboyadzhi/triathlon-results")
tri_files = glob.glob(os.path.join(path_tri, '*.csv'))
df_triathlon = pd.concat([pd.read_csv(f) for f in tri_files], ignore_index=True)
print(f"Triathlon: {df_triathlon.shape[0]:,} filas x {df_triathlon.shape[1]} columnas")
print("Columnas:", df_triathlon.columns.tolist()[:15])

In [ ]:
# 1.2 Athlete training and recovery tracker (solo exploración)
path_ath = kagglehub.dataset_download("prince7489/athlete-training-and-recovery-tracker-dataset")
ath_files = glob.glob(os.path.join(path_ath, '*.csv'))
df_recovery = pd.concat([pd.read_csv(f) for f in ath_files], ignore_index=True)
print(f"Recovery: {df_recovery.shape[0]:,} filas x {df_recovery.shape[1]} columnas")

# Los nombres de columna cambian entre versiones del conjunto,
# así que se normalizan buscando palabras clave
col_map = {}
for col in df_recovery.columns:
    cl = col.lower().replace(' ', '_')
    if ('heart' in cl or cl in ('hr', 'avg_hr')): col_map[col] = 'avg_hr'
    elif 'session' in cl and 'load' in cl: col_map[col] = 'session_load'
    elif 'recovery' in cl and 'score' in cl: col_map[col] = 'recovery_score'
    elif 'sleep' in cl and ('hour' in cl or 'qual' in cl): col_map[col] = 'sleep_quality'
    elif 'stress' in cl: col_map[col] = 'stress_score'
    elif 'duration' in cl: col_map[col] = 'duration_min'
    elif 'distance' in cl: col_map[col] = 'distance_km'
    elif any(x in cl for x in ('sport', 'activity', 'exercise')): col_map[col] = 'sport_type'
    elif 'date' in cl: col_map[col] = 'date'
    elif 'hrv' in cl: col_map[col] = 'hrv'
df_recovery_clean = df_recovery.rename(columns=col_map)
print(f"{len(df_recovery_clean)} sesiones de recovery")

In [ ]:
# 1.3 GoldenCheetah OpenData (solo exploración). Si la descarga falla se sigue
# con tablas vacías: ninguna fase posterior depende de este conjunto.
def _leer_tabla(path):
    """Lee un CSV; si 'path' es una carpeta de partes, concatena los CSV internos."""
    if os.path.isdir(path):
        partes = [p for p in glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
                  if os.path.isfile(p)]
        return pd.concat([pd.read_csv(p) for p in partes], ignore_index=True)
    return pd.read_csv(path)

def _localizar(path_base, clave, excluir=()):
    """Devuelve la ruta (archivo o carpeta) cuyo nombre contiene 'clave'."""
    cand = [p for p in glob.glob(os.path.join(path_base, "**", "*"), recursive=True)
            if clave in os.path.basename(p).lower()
            and not any(x in os.path.basename(p).lower() for x in excluir)]
    archivos = [p for p in cand if os.path.isfile(p)]   # prioriza archivos reales
    return (archivos or cand or [None])[0]

try:
    path_gc = kagglehub.dataset_download(
        "markliversedge/goldencheetah-opendata-athlete-activity-and-mmp")
    ruta_act = _localizar(path_gc, "activit", excluir=("mmp",))   # actividades, sin MMP
    ruta_ath = _localizar(path_gc, "athlet")                       # atletas
    df_gc = _leer_tabla(ruta_act) if ruta_act else pd.DataFrame()
    df_gc_ath = _leer_tabla(ruta_ath) if ruta_ath else pd.DataFrame()
    print(f"GoldenCheetah REAL · actividades: {df_gc.shape} · atletas: {df_gc_ath.shape}")
    print("Columnas actividades:", list(df_gc.columns)[:25])
    print("Columnas atletas:", list(df_gc_ath.columns)[:25])
except Exception as e:
    print(f"No se pudo cargar GoldenCheetah ({e}); revisa kaggle.json")
    df_gc = pd.DataFrame(); df_gc_ath = pd.DataFrame()

In [ ]:
print("=== df_triathlon (resultados reales) ===  filas:", len(df_triathlon))
print(list(df_triathlon.columns))
print(df_triathlon.head(3).to_string())

print("\n=== df_gc (GoldenCheetah) ===")
try:
    print("filas:", len(df_gc), "| columnas:", list(df_gc.columns))
    if 'sport' in df_gc.columns: print(df_gc['sport'].value_counts())
except Exception as e:
    print("df_gc no disponible:", e)

## 2. Carga de los datos personales (Garmin Connect)

Cada deportista cedió la exportación completa de su cuenta de Garmin Connect (un ZIP por persona).
El cuaderno busca los ZIP en la carpeta actual, en `/content/` y en Google Drive
(`MyDrive/TFM_Triatlon/` o `MyDrive/`). Para subirlos directamente a Colab:

```python
from google.colab import files
files.upload()   # Josh.zip, Marco.zip, Andresin.zip, Patico.zip
```

In [ ]:
# Funciones auxiliares para dos campos de Garmin que vienen anidados

def extract_body_battery(bb_raw):
    """Saca el Body Battery de inicio de día del JSON de Garmin."""
    if isinstance(bb_raw, (int, float)):
        return float(bb_raw)
    if isinstance(bb_raw, dict):
        stat_list = bb_raw.get('bodyBatteryStatList', [])
        if stat_list:
            stats = {s['bodyBatteryStatType']: s['statsValue'] for s in stat_list}
            return float(stats.get('STARTOFDAY',
                         stats.get('HIGHEST',
                         stats.get('MOSTRECENT', float('nan')))))
        cv = bb_raw.get('chargedValue')
        if cv is not None:
            return float(cv)
    return float('nan')

def extract_stress(stress_raw):
    """Nivel de estrés medio del día. Garmin usa dos formatos distintos."""
    if isinstance(stress_raw, (int, float)):
        return float(stress_raw)
    if isinstance(stress_raw, dict):
        agg_list = stress_raw.get('aggregatorList', [])
        if agg_list:
            total = next((a for a in agg_list if a.get('type') == 'TOTAL'), None)
            if total:
                return float(total.get('averageStressLevel', float('nan')))
        return float(stress_raw.get('avgStressLevel', stress_raw.get('average', float('nan'))))
    return float('nan')

print("Funciones auxiliares listas")

In [ ]:
# Lector principal: del ZIP de Garmin Connect a tablas de pandas.
# Extrae actividades, bienestar diario, sueño, readiness/HRV y VO2max.
# Si falta una variable se registra como ausente y el proceso continúa
# (memoria, apartados 3.2 y 4.2).

def parse_garmin_zip(zip_path, athlete_name):
    data = {k: [] for k in ('activities', 'uds', 'sleep', 'readiness', 'vo2')}

    with zipfile.ZipFile(zip_path, 'r') as z:
        files = z.namelist()

        # Actividades resumidas
        for af in [f for f in files if 'summarizedActivities' in f and f.endswith('.json')]:
            with z.open(af) as f: raw = json.load(f)
            acts = raw[0]['summarizedActivitiesExport'] if isinstance(raw, list) else raw.get('summarizedActivitiesExport', [])
            for a in acts:
                sport = a.get('sportType', '')
                if sport not in ('RUNNING', 'CYCLING', 'SWIMMING'): continue
                ts = a.get('startTimeGmt', 0)
                date = pd.to_datetime(ts, unit='ms', utc=True).date() if ts else None
                dur_ms    = a.get('duration') or a.get('movingDuration', 0)
                dist_cm   = a.get('distance', np.nan)
                speed_raw = a.get('avgSpeed', np.nan)
                speed_kmh = speed_raw * 36 if speed_raw else np.nan
                # por debajo de 5 km/h suele ser rodillo sin GPS: se marca como ausente
                if sport == 'CYCLING' and (not speed_kmh or speed_kmh < 5):
                    speed_kmh = np.nan
                elev_cm      = a.get('elevationGain', np.nan)
                elev_loss_cm = a.get('elevationLoss', np.nan)
                start_alt_cm = a.get('startElevation', np.nan)
                cadence = a.get('avgDoubleCadence') or ((a.get('avgRunCadence', 0) or 0) * 2) or np.nan
                data['activities'].append({
                    'athlete': athlete_name, 'date': date, 'sport': sport,
                    'duration_min':   round(dur_ms / 60000, 1),
                    'distance_km':    round(dist_cm / 100000, 2) if dist_cm else np.nan,
                    'avg_speed_kmh':  round(speed_kmh, 2) if not pd.isna(speed_kmh) else np.nan,
                    'avg_hr':         a.get('avgHr', np.nan),
                    'max_hr':         a.get('maxHr', np.nan),
                    'calories':       a.get('calories', np.nan),
                    'elevation_m':       round(elev_cm / 100, 1) if elev_cm else np.nan,
                    'elevation_loss_m':  round(elev_loss_cm / 100, 1) if elev_loss_cm else np.nan,
                    'start_altitude_m':  round(start_alt_cm / 100, 1) if start_alt_cm else np.nan,
                    'cadence':        cadence,
                    'stride_length_cm': a.get('avgStrideLength', np.nan),
                    'aerobic_te':     a.get('aerobicTrainingEffect', np.nan),
                    'anaerobic_te':   a.get('anaerobicTrainingEffect', np.nan),
                    'training_load':  a.get('activityTrainingLoad', np.nan),
                    'avg_respiration': a.get('avgRespirationRate', np.nan),
                    'min_temp':       a.get('minTemperature', np.nan),
                    'max_temp':       a.get('maxTemperature', np.nan),
                    'vo2max_activity': a.get('vO2MaxValue', np.nan),
                })

        # Bienestar diario (archivos UDS)
        for uf in [f for f in files if 'UDSFile' in f and f.endswith('.json')]:
            with z.open(uf) as f: uds_raw = json.load(f)
            for r in uds_raw:
                ds = r.get('calendarDate', '')
                if not ds: continue
                data['uds'].append({
                    'athlete': athlete_name,
                    'date':         pd.to_datetime(ds).date(),
                    'resting_hr':   r.get('restingHeartRate', np.nan),
                    'min_hr':       r.get('minHeartRate', np.nan),
                    'max_hr_day':   r.get('maxHeartRate', np.nan),
                    'body_battery': extract_body_battery(r.get('bodyBattery', {})),
                    'stress_avg':   extract_stress(r.get('allDayStress', {})),
                    'active_seconds': r.get('activeSeconds', np.nan),
                    'steps':        r.get('totalSteps', np.nan),
                    'distance_day_m': r.get('totalDistanceMeters', np.nan),
                })

        # Sueño
        for sf in [f for f in files if 'sleepData' in f and f.endswith('.json')]:
            with z.open(sf) as f: sleep_raw = json.load(f)
            for r in sleep_raw:
                ds = r.get('calendarDate', '')
                if not ds: continue
                deep  = r.get('deepSleepSeconds', 0) or 0
                light = r.get('lightSleepSeconds', 0) or 0
                rem   = r.get('remSleepSeconds', 0) or 0
                total = deep + light + rem
                data['sleep'].append({
                    'athlete': athlete_name,
                    'date': pd.to_datetime(ds).date(),
                    'sleep_total_min': round(total / 60, 1),
                    'sleep_deep_min':  round(deep / 60, 1),
                    'sleep_rem_min':   round(rem / 60, 1),
                    'sleep_deep_pct':  round(deep / total * 100, 1) if total > 0 else np.nan,
                })

        # Training readiness (incluye HRV semanal)
        for rf in [f for f in files if 'TrainingReadiness' in f and f.endswith('.json')]:
            with z.open(rf) as f: read_raw = json.load(f)
            for r in read_raw:
                ds = r.get('calendarDate', '')
                if not ds: continue
                data['readiness'].append({
                    'athlete': athlete_name,
                    'date': pd.to_datetime(ds).date(),
                    'readiness_score':    r.get('score', np.nan),
                    'readiness_level':    r.get('level', ''),
                    'sleep_score':        r.get('sleepScore', np.nan),
                    'hrv_weekly_avg':     r.get('hrvWeeklyAverage', np.nan),
                    'recovery_time_h':    r.get('recoveryTime', np.nan),
                    'acute_load':         r.get('acuteLoad', np.nan),
                    'hrv_factor_pct':     r.get('hrvFactorPercent', np.nan),
                    'stress_history_pct': r.get('stressHistoryFactorPercent', np.nan),
                })

        # VO2max
        for vf in [f for f in files if ('Vo2Max' in f or 'vo2' in f.lower()) and f.endswith('.json')]:
            with z.open(vf) as f: vo2_raw = json.load(f)
            for r in vo2_raw:
                ds = r.get('calendarDate', '')
                if not ds: continue
                data['vo2'].append({
                    'athlete': athlete_name,
                    'date': pd.to_datetime(ds).date(),
                    'sport': r.get('sport', ''),
                    'vo2max': r.get('vo2MaxValue', np.nan),
                })

    result = {}
    for key, rows in data.items():
        if rows:
            df = pd.DataFrame(rows)
            df['date'] = pd.to_datetime(df['date'])
            result[key] = df.sort_values('date').reset_index(drop=True)
        else:
            result[key] = pd.DataFrame()

    # resumen rápido por consola para validar la carga
    acts = result['activities']
    sport_counts = acts.groupby('sport').size().to_dict() if not acts.empty else {}
    n_uds = len(result['uds']); n_sl = len(result['sleep']); n_rd = len(result['readiness'])
    bb_avg = result['uds']['body_battery'].mean() if not result['uds'].empty else np.nan
    fc_rep = result['uds']['resting_hr'].median() if not result['uds'].empty else np.nan
    print(f"  {athlete_name:10s}: bici={sport_counts.get('CYCLING', 0)} carrera={sport_counts.get('RUNNING', 0)} "
          f"natacion={sport_counts.get('SWIMMING', 0)} | BB={bb_avg:.0f} | FCrep={fc_rep:.0f} | "
          f"bienestar={n_uds}d sueno={n_sl}d readiness={n_rd}d")
    return result

print("Lector definido")

In [ ]:
# Localiza los ZIP y los procesa. Fuera de Colab basta con dejarlos en la carpeta del cuaderno.
SEARCH_PATHS = ['.', '/content/', '/content/drive/MyDrive/TFM_Triatlon/', '/content/drive/MyDrive/']

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
except ImportError:
    pass   # no estamos en Colab

def find_zip(nombre):
    for base in SEARCH_PATHS:
        encontrados = glob.glob(os.path.join(base, f'{nombre}*.zip'))
        if encontrados:
            return encontrados[0]
    return None

garmin_data = {}
print("Cargando deportistas...")
for letra, nombre_zip in PARTICIPANTES.items():
    ruta = find_zip(nombre_zip)
    if ruta:
        garmin_data[letra] = parse_garmin_zip(ruta, letra)
    else:
        print(f"  {letra}: no se encontró {nombre_zip}.zip, se omite")
print(f"\n{len(garmin_data)} deportistas cargados")

## 3. Fase 0 — Actividades, TRIMP y modelo Fitness-Fatiga (CTL, ATL y TSB)

Memoria, apartados 3.3, 3.5 y 5.1. El TRIMP de Banister mide la carga de cada sesión; con la serie diaria
se calculan la carga crónica (CTL, forma), la aguda (ATL, fatiga) y su diferencia (TSB, frescura) mediante
medias móviles exponenciales de 42 y 7 días.

In [ ]:
# Consolidar las actividades de todos en un solo DataFrame
dfs = [d['activities'] for d in garmin_data.values() if not d.get('activities', pd.DataFrame()).empty]
df_act = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"Total actividades: {len(df_act):,}")
if not df_act.empty:
    print(df_act.groupby(['athlete', 'sport']).size().unstack(fill_value=0).to_string())

In [ ]:
resting_hrs = {
    name: d['uds']['resting_hr'].median() if not d.get('uds', pd.DataFrame()).empty else 55
    for name, d in garmin_data.items()
}

# El sexo de cada deportista (sexo_por_atleta) está en la celda de configuración.

def calcular_trimp(row, fc_max=185, sexo='M'):
    """TRIMP de Banister con coeficientes por sexo y ajuste por desnivel (memoria, 3.5).

    Coeficientes de Banister (distinta respuesta del lactato):
      - hombres: 0,64 · e^(1,92 · r)
      - mujeres: 0,86 · e^(1,67 · r)
    Las sesiones con más de 200 m de desnivel positivo multiplican su carga por
    (1 + 0,0001 · (D+ − 200)). Es un ajuste propio del sistema para reflejar el
    esfuerzo de las subidas. La FC máxima se fija en 185 ppm para todos, una
    simplificación que la memoria declara en las limitaciones (6.2).
    """
    dur = row.get('duration_min', np.nan)
    fc  = row.get('avg_hr', np.nan)
    if pd.isna(dur) or pd.isna(fc) or dur <= 0: return np.nan
    ratio = np.clip((fc - row.get('fc_rep', 55)) / (fc_max - row.get('fc_rep', 55)), 0.01, 1.0)
    coef, k = (0.64, 1.92) if str(sexo).upper().startswith('M') else (0.86, 1.67)
    trimp_base = dur * ratio * coef * np.exp(k * ratio)
    elev = row.get('elevation_m', 0) or 0
    elev_factor = 1 + max(0, elev - 200) * 0.0001
    return round(trimp_base * elev_factor, 2)

def corregir_vo2_altitud(vo2_raw, altitude_m):
    """Corrección aproximada del VO2máx del reloj para quien entrena en altura:
    se resta un 1 % por cada 100 m por encima de 1.500 m. Solo se usa para
    describir al deportista; ningún modelo utiliza el VO2máx.
    """
    if pd.isna(vo2_raw) or pd.isna(altitude_m) or altitude_m < 1500:
        return vo2_raw
    penalty = (altitude_m - 1500) / 100 * 0.01
    return round(float(vo2_raw) * (1 - penalty), 1)

if not df_act.empty:
    df_act['fc_rep'] = df_act['athlete'].map(resting_hrs).fillna(55)
    df_act['sexo'] = df_act['athlete'].map(sexo_por_atleta).fillna('M')
    df_act['TRIMP'] = df_act.apply(lambda r: calcular_trimp(r, sexo=r['sexo']), axis=1)
    print(f"TRIMP calculado en {df_act['TRIMP'].notna().sum():,} de {len(df_act):,} actividades")
else:
    print("No hay actividades: revisa que se hayan cargado los ZIP de Garmin.")
    df_act['fc_rep'] = np.nan
    df_act['sexo'] = 'M'
    df_act['TRIMP'] = np.nan

In [ ]:
# CTL/ATL/TSB con medias móviles exponenciales (42 y 7 días)
def calcular_fitness_fatiga(df_athlete):
    if df_athlete.empty: return pd.DataFrame()
    daily = df_athlete.groupby('date')['TRIMP'].sum().reset_index()
    daily['date'] = pd.to_datetime(daily['date'])
    idx   = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')
    daily = daily.set_index('date').reindex(idx, fill_value=0).reset_index()
    daily.columns = ['date', 'trimp_daily']
    daily['CTL'] = daily['trimp_daily'].ewm(span=42, adjust=False).mean().round(1)
    daily['ATL'] = daily['trimp_daily'].ewm(span=7,  adjust=False).mean().round(1)
    daily['TSB'] = (daily['CTL'] - daily['ATL']).round(1)
    return daily

fitness_data = {}
print("Estado fitness-fatiga actual:")
for name in garmin_data:
    acts_a = df_act[df_act['athlete'] == name]
    fd = calcular_fitness_fatiga(acts_a)
    if not fd.empty:
        fd['athlete'] = name
        fitness_data[name] = fd
        nota = " (caso de evaluación)" if name == SUJETO_PRUEBA else ""
        print(f"  {name:10s}: CTL={fd['CTL'].iloc[-1]:.1f} ATL={fd['ATL'].iloc[-1]:.1f} TSB={fd['TSB'].iloc[-1]:.1f}{nota}")

In [ ]:
# Gráfico CTL/ATL/TSB por participante (último año)
fig = make_subplots(rows=len(fitness_data), cols=1,
    subplot_titles=[f"Atleta {n} — Fitness-Fatiga" for n in fitness_data],
    shared_xaxes=False, vertical_spacing=0.08)

for i, (name, fd) in enumerate(fitness_data.items(), 1):
    fd_plot = fd.tail(365)
    fig.add_trace(go.Scatter(x=fd_plot['date'], y=fd_plot['CTL'],
        name='CTL', line=dict(color='#2196F3', width=2), showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=fd_plot['date'], y=fd_plot['ATL'],
        name='ATL', line=dict(color='#F44336', width=1.5, dash='dash'), showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=fd_plot['date'], y=fd_plot['TSB'],
        name='TSB', line=dict(color='#4CAF50', width=1.5, dash='dot'),
        fill='tozeroy', fillcolor='rgba(76,175,80,0.1)', showlegend=(i == 1)), row=i, col=1)
    # referencias: 0, zona de riesgo (-25) y frescura (+10)
    fig.add_hline(y=0,   line_dash='dash', line_color='gray',    opacity=0.5, row=i, col=1)
    fig.add_hline(y=-25, line_dash='dot',  line_color='#F44336', opacity=0.6, row=i, col=1)
    fig.add_hline(y=10,  line_dash='dot',  line_color='#4CAF50', opacity=0.6, row=i, col=1)

fig.update_layout(height=280 * len(fitness_data), title_text="Modelo Fitness-Fatiga (Banister)",
                  template='plotly_white', hovermode='x unified')
fig.show()

In [ ]:
# Tabla 11 de la memoria: cobertura de los datos por deportista. El periodo ignora
# la actividad con fecha de 1989 de C, un error de registro (memoria, 5.1).
filas = []
for letra, d in garmin_data.items():
    a = df_act[df_act['athlete'] == letra]
    n = a['sport'].value_counts()
    uds, rd, sl = d.get('uds', pd.DataFrame()), d.get('readiness', pd.DataFrame()), d.get('sleep', pd.DataFrame())
    metricas = []
    if not uds.empty and uds['body_battery'].notna().any(): metricas.append('Body Battery')
    if not uds.empty and uds['stress_avg'].notna().any(): metricas.append('estrés')
    if not sl.empty: metricas.append('sueño')
    if not rd.empty and rd['hrv_weekly_avg'].notna().any(): metricas.append('HRV')
    filas.append({'Atleta': letra,
                  'Actividades (nat / bici / carrera)': f"{len(a)} ({n.get('SWIMMING', 0)} / {n.get('CYCLING', 0)} / {n.get('RUNNING', 0)})",
                  'Periodo': f"{a.loc[a['date'].dt.year >= 2000, 'date'].min().year}-{a['date'].max().year}",
                  'Actividades con FC (%)': round(a['avg_hr'].notna().mean() * 100, 1),
                  'Métricas de recuperación': ', '.join(metricas) or 'ninguna'})
tabla_11 = pd.DataFrame(filas).set_index('Atleta')
print(f"TRIMP calculable en {df_act['TRIMP'].notna().sum():,} de {len(df_act):,} actividades "
      f"({df_act['TRIMP'].notna().mean() * 100:.1f} %)")
tabla_11

In [ ]:
# Figuras 9 y 12 de la memoria
letras = list(garmin_data)
etiqueta = {l: f'Atleta {l}' for l in letras}

# Figura 9. Actividades registradas por deportista y disciplina
ct = df_act.groupby(['athlete', 'sport']).size().unstack(fill_value=0).reindex(letras)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
colores = {'SWIMMING': '#4C72B0', 'CYCLING': '#DD8452', 'RUNNING': '#55A868'}
nombres = {'SWIMMING': 'Natación', 'CYCLING': 'Ciclismo', 'RUNNING': 'Carrera'}
izq = np.zeros(len(letras))
for sp in ['SWIMMING', 'CYCLING', 'RUNNING']:
    v = ct.get(sp, pd.Series(0, index=ct.index)).values
    ax.barh([etiqueta[l] for l in letras], v, left=izq, color=colores[sp], label=nombres[sp], edgecolor='white')
    for i, (l0, val) in enumerate(zip(izq, v)):
        if val >= 60:
            ax.text(l0 + val / 2, i, str(val), ha='center', va='center', fontsize=9)
    izq += v
for i, t in enumerate(izq):
    ax.text(t + 15, i, f'{int(t)}', va='center', fontsize=9, fontweight='bold')
ax.invert_yaxis(); ax.set_xlabel('Número de actividades registradas')
ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.2), frameon=False); ax.set_xlim(0, max(izq) * 1.12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig('salidas/figuras/fig09_actividades.png', dpi=200); plt.show()

# Figura 12. CTL, ATL y TSB del último año de cada deportista
fig, axs = plt.subplots(2, 2, figsize=(8, 5.2))
for ax, l in zip(axs.flat, letras):
    d = fitness_data[l]
    d = d[d['date'] >= d['date'].max() - pd.Timedelta(days=365)]
    ax.plot(d['date'], d['CTL'], color='#1f4e79', lw=1.6, label='CTL (forma, 42 d)')
    ax.plot(d['date'], d['ATL'], color='#c0504d', lw=1.0, label='ATL (fatiga, 7 d)')
    ax.fill_between(d['date'], d['TSB'], 0, where=d['TSB'] < 0, color='#f4b183', alpha=0.5, label='TSB < 0 (fatiga)')
    ax.fill_between(d['date'], d['TSB'], 0, where=d['TSB'] >= 0, color='#a9d18e', alpha=0.6, label='TSB ≥ 0 (frescura)')
    ax.axhline(0, color='black', lw=0.6); ax.set_title(etiqueta[l], fontsize=10)
    ax.tick_params(axis='x', labelsize=8, rotation=30); ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylabel('TRIMP/día', fontsize=9)
h, lab = axs[0, 0].get_legend_handles_labels()
fig.legend(h, lab, ncol=4, loc='lower center', frameon=False, fontsize=8.5)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.savefig('salidas/figuras/fig12_ctl_atl_tsb.png', dpi=200); plt.show()

**Hallazgos de la fase 0.** Se procesan 3.771 actividades (1.616 de ciclismo, 1.513 de carrera y 642 de
natación), pero la carga TRIMP solo se puede calcular en 2.436 (64,6 %), porque falta la frecuencia cardíaca.
El caso extremo es D, con FC en el 8,7 % de sus actividades: su CTL es casi nula y cualquier decisión basada
en la carga no es fiable para él. Los historiales son muy desiguales: A apenas tiene 15 sesiones de ciclismo,
que es justo la situación que debe resolver la fase 1. La revisión detecta además registros erróneos que las
reglas de limpieza no filtran, como una actividad fechada en 1989.

## 4. Variables de cada deportista en los últimos 90 días

Memoria, apartado 3.3. Para cada deportista se resumen los últimos 90 días en siete categorías: sesión,
intensidad, rendimiento, fatiga, recuperación, técnica y ambiente (34 variables). Describen su estado
reciente y aportan la FC de reposo y las velocidades de referencia con las que se calculan sus zonas y
ritmos (apartado 4.5.1).

In [ ]:
def build_athlete_profile(name):
    """Construye el vector de variables de un deportista. Si una variable no está
    disponible queda como ausente y el proceso continúa."""
    p = {'athlete': name}
    acts = df_act[df_act['athlete'] == name].copy()
    if acts.empty: return p

    cutoff = acts['date'].max() - pd.Timedelta(days=90)
    recent = acts[acts['date'] >= cutoff]
    if recent.empty: recent = acts.tail(30)

    # 1. Sesión
    p['sessions_per_week']  = len(recent) / 12.86   # 90 dias / 7
    p['avg_duration_min']   = recent['duration_min'].mean()
    for sp in ('RUNNING', 'CYCLING', 'SWIMMING'):
        p[f'sessions_{sp.lower()}'] = (recent['sport'] == sp).sum()

    # 2. Intensidad
    p['avg_hr']             = recent['avg_hr'].mean()
    p['avg_trimp_session']  = recent['TRIMP'].mean()
    p['weekly_trimp']       = recent['TRIMP'].sum() / 12.86
    fc_max = 185
    p['pct_z12'] = len(recent[recent['avg_hr'] < fc_max * 0.75]) / len(recent) * 100 if len(recent) else np.nan
    p['pct_z45'] = len(recent[recent['avg_hr'] > fc_max * 0.85]) / len(recent) * 100 if len(recent) else np.nan

    # 3. Rendimiento por disciplina
    run = recent[recent['sport'] == 'RUNNING']
    cyc = recent[recent['sport'] == 'CYCLING'].dropna(subset=['avg_speed_kmh'])
    swm = recent[recent['sport'] == 'SWIMMING']
    p['pace_running_kmh']  = run['avg_speed_kmh'].mean() if not run.empty else np.nan
    p['pace_cycling_kmh']  = cyc['avg_speed_kmh'].mean() if not cyc.empty else np.nan
    p['pace_swimming_kmh'] = swm['avg_speed_kmh'].mean() if not swm.empty else np.nan
    alt_median = (recent['start_altitude_m'].median()
                  if 'start_altitude_m' in recent.columns else np.nan)
    vo2_raw = recent['vo2max_activity'].dropna().mean()
    p['vo2max']            = corregir_vo2_altitud(vo2_raw, alt_median)
    p['train_altitude_m']  = alt_median

    # 4. Fatiga / HRV
    d = garmin_data.get(name, {})
    rd = d.get('readiness', pd.DataFrame())
    if not rd.empty:
        rd_r = rd.tail(90)
        p['hrv_weekly_avg']      = rd_r['hrv_weekly_avg'].median()
        p['hrv_factor_pct']      = rd_r['hrv_factor_pct'].median()
        p['readiness_score_avg'] = rd_r['readiness_score'].mean()
        p['recovery_time_avg_h'] = rd_r['recovery_time_h'].median()
    else:
        p['hrv_weekly_avg'] = p['hrv_factor_pct'] = p['readiness_score_avg'] = p['recovery_time_avg_h'] = np.nan

    # 5. Recuperación
    uds = d.get('uds', pd.DataFrame())
    if not uds.empty:
        uds_r = uds.tail(90)
        p['resting_hr']       = uds_r['resting_hr'].median()
        p['body_battery_avg'] = uds_r['body_battery'].mean()
        p['stress_avg']       = uds_r['stress_avg'].mean()
    else:
        p['resting_hr'] = p['body_battery_avg'] = p['stress_avg'] = np.nan

    sl = d.get('sleep', pd.DataFrame())
    if not sl.empty:
        sl_r = sl.tail(90)
        p['sleep_total_avg_min'] = sl_r['sleep_total_min'].mean()
        p['sleep_deep_pct']      = sl_r['sleep_deep_pct'].mean()
    else:
        p['sleep_total_avg_min'] = p['sleep_deep_pct'] = np.nan

    # 6. Técnica
    p['avg_cadence']    = recent['cadence'].mean()
    p['avg_stride_cm']  = run['stride_length_cm'].mean() if not run.empty else np.nan

    # 7. Ambiente: desnivel (D+ carga cardiovascular, D- carga excéntrica)
    # y altitud media, que condiciona el VO2max
    p['avg_elevation_gain_m'] = recent['elevation_m'].mean()
    p['avg_elevation_loss_m'] = (recent['elevation_loss_m'].mean()
                                  if 'elevation_loss_m' in recent.columns else np.nan)
    p['avg_start_altitude_m'] = (recent['start_altitude_m'].mean()
                                  if 'start_altitude_m' in recent.columns else np.nan)
    p['avg_temp'] = ((recent['min_temp'].fillna(0) + recent['max_temp'].fillna(0)) / 2).mean()

    # Estado fitness-fatiga actual
    if name in fitness_data:
        fd = fitness_data[name]
        p['ctl_current'] = fd['CTL'].iloc[-1]
        p['atl_current'] = fd['ATL'].iloc[-1]
        p['tsb_current'] = fd['TSB'].iloc[-1]
        p['ctl_trend_30d'] = fd['CTL'].iloc[-1] - fd['CTL'].iloc[-30]
    else:
        p['ctl_current'] = p['atl_current'] = p['tsb_current'] = p['ctl_trend_30d'] = np.nan

    return p

athlete_profiles = [build_athlete_profile(n) for n in garmin_data]
df_profiles = pd.DataFrame(athlete_profiles).set_index('athlete')
print(f"Perfiles construidos: {df_profiles.shape}")
df_profiles.round(2)

## 5. Análisis exploratorio descartado: agrupamiento K-means

Durante el desarrollo se probó un agrupamiento K-means para asignar un perfil a cada deportista. Se descartó
(memoria, apartado 6.2): con cuatro casos hay que añadir una población de referencia sintética de 300
perfiles, los grupos no se separan (coeficiente de silueta de 0,06 con k = 4) y el perfil no modifica el plan.

Se conserva solo como registro. Ningún resultado de esta sección se usa en el resto del cuaderno, se guarda
en `salidas/` ni lo utiliza la aplicación.

In [ ]:
# Variables que entraban en el agrupamiento
cluster_features = ['sessions_per_week', 'weekly_trimp', 'pct_z12', 'pct_z45', 'avg_hr',
                    'resting_hr', 'body_battery_avg', 'stress_avg', 'sleep_deep_pct',
                    'avg_cadence', 'pace_running_kmh', 'ctl_current', 'tsb_current', 'vo2max']
cluster_features = [f for f in cluster_features if f in df_profiles.columns]
X_cluster = df_profiles[cluster_features]

# Con cuatro deportistas no se puede agrupar, así que se añade una población
# de referencia sintética de 300 perfiles con rangos orientados por los
# conjuntos públicos. Esa población es la principal debilidad del análisis.
np.random.seed(42); n_pub = 300
df_pub = pd.DataFrame({
    'sessions_per_week':  np.random.normal(5, 2, n_pub).clip(1, 14),
    'weekly_trimp':       np.random.normal(250, 100, n_pub).clip(20, 800),
    'avg_hr':             np.random.normal(148, 15, n_pub).clip(90, 185),
    'pct_z12':            np.random.normal(65, 15, n_pub).clip(0, 100),
    'pct_z45':            np.random.normal(10, 8, n_pub).clip(0, 100),
    'pace_running_kmh':   np.random.normal(10, 2, n_pub).clip(5, 18),
    'ctl_current':        np.random.normal(45, 20, n_pub).clip(3, 120),
    'tsb_current':        np.random.normal(-5, 15, n_pub).clip(-50, 30),
    'resting_hr':         np.random.normal(58, 8, n_pub).clip(40, 80),
    'body_battery_avg':   np.random.normal(55, 18, n_pub).clip(10, 95),
}).reindex(columns=cluster_features)

df_all_c = pd.concat([X_cluster, df_pub], ignore_index=True)
imp = SimpleImputer(strategy='median'); sc = StandardScaler()
X_all   = sc.fit_transform(imp.fit_transform(df_all_c[cluster_features]))
X_pers  = sc.transform(imp.transform(X_cluster[cluster_features]))

# Coeficiente de silueta para k entre 2 y 6
k_range = range(2, min(7, len(df_all_c)))
sil_scores = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    sil_scores.append(silhouette_score(X_all, km.fit_predict(X_all)))

best_k = 4   # número de grupos fijado en el diseño explorado; la silueta máxima está en k = 2
print(f"Silueta por k: {dict(zip(k_range, [round(s, 3) for s in sil_scores]))}")
print(f"k usado: {best_k} (silueta {sil_scores[list(k_range).index(best_k)]:.3f})")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(list(k_range), sil_scores, 'bo-', markersize=8)
ax.axvline(best_k, color='red', linestyle='--', label=f'k={best_k}')
ax.set_xlabel('Número de grupos (k)'); ax.set_ylabel('Coeficiente de silueta')
ax.set_title('Agrupamiento descartado: silueta según k')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Grupo asignado a cada deportista. Se guarda aparte para no mezclarlo con
# las variables que sí se usan (df_profiles).
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
kmeans.fit(X_all)
grupo_kmeans = pd.Series(kmeans.predict(X_pers), index=X_cluster.index, name='grupo')
print("Grupo de cada deportista (solo informativo, no se usa):")
print(grupo_kmeans.to_string())

**Hallazgo.** La silueta máxima (0,81) aparece con k = 2 y solo separa a D, cuyo vector está formado casi
por completo por valores imputados. Con k entre 3 y 6 no supera 0,08: los grupos no están separados. Por
eso el agrupamiento se descartó y ningún resultado de esta sección se usa después.

## 6. Fase 1 — Transferencia entre disciplinas (XGBoost)

Memoria, apartados 3.4, 4.4 y 5.2. Dos modelos estiman la velocidad en la disciplina que el deportista no
ha entrenado:

- **Modelo directo:** velocidad en ciclismo a partir de la velocidad de carrera y el sexo.
- **Modelo inverso:** velocidad de carrera a partir de la velocidad en ciclismo y el sexo.

Se entrenan con 80.000 triatletas de resultados reales de 70.3. Los cuatro deportistas no intervienen en
el entrenamiento: solo se les aplica el modelo. La velocidad de natación no se usa como predictora
(apartado 3.4); la sección 6c mide cuánto se pierde al retirarla.

El deportista A tiene 252 sesiones de carrera y solo 15 de ciclismo: es el caso para el que se pensó el
modelo directo.

In [ ]:
# Preparación de los datos de entrenamiento: tiempos parciales reales de pruebas 70.3.
# La velocidad de natación se calcula solo para la comprobación de la sección 6c.

def _t2min(txt):
    if pd.isna(txt): return np.nan
    s = str(txt).strip()
    if not s or any(c.isalpha() for c in s): return np.nan
    p = s.split(':')
    try: p = [float(x) for x in p]
    except ValueError: return np.nan
    if len(p) == 3: return p[0]*60 + p[1] + p[2]/60
    if len(p) == 2: return p[0] + p[1]/60
    return np.nan

half = df_triathlon[
    df_triathlon['event_link'].astype(str).str.contains('/half/', na=False) |
    df_triathlon['event_name'].astype(str).str.contains('70.3', na=False)].copy()
for c_src, c_min in [('person_event_cycle_time_text', 'bike_min'),
                     ('person_event_run_time_text',  'run_min'),
                     ('person_event_swim_time_text', 'swim_min')]:
    half[c_min] = half[c_src].apply(_t2min)
half = half.dropna(subset=['bike_min', 'run_min'])
half = half[(half['bike_min'] > 0) & (half['run_min'] > 0)]
half['pace_cycling_kmh']  = 90.0 / (half['bike_min'] / 60.0)
half['running_speed_kmh'] = 21.1 / (half['run_min']  / 60.0)
half['swim_speed_kmh']    = 1.9  / (half['swim_min'] / 60.0)
half[['pace_cycling_kmh', 'running_speed_kmh', 'swim_speed_kmh']] = \
    half[['pace_cycling_kmh', 'running_speed_kmh', 'swim_speed_kmh']].replace([np.inf, -np.inf], np.nan)
half = half[half['pace_cycling_kmh'].between(15, 50) &
            half['running_speed_kmh'].between(5, 20)]
g = half['gender'].astype(str).str.upper().str[0]
half['sexo_M'] = np.where(g == 'M', 1.0, np.where(g.isin(['F', 'W']), 0.0, np.nan))
print(f"Registros válidos de 70.3 con ciclismo y carrera: {len(half):,}")
print(half[['pace_cycling_kmh','running_speed_kmh','swim_speed_kmh','sexo_M']].describe().round(2))

# Velocidad sostenida: percentil 80 de las últimas 25 sesiones (las actividades
# están ordenadas por fecha). La media incluiría los rodajes suaves y
# subestimaría lo que el deportista puede sostener.
def ritmo_sostenido(df_sport, n=25, q=0.80):
    if df_sport.empty:
        return np.nan
    return df_sport.tail(n)['avg_speed_kmh'].quantile(q)

# Hiperparámetros fijados a priori, sin búsqueda (memoria, 3.4). Tasa de aprendizaje
# baja con más árboles y un 80 % de las filas por árbol, las dos formas de
# regularización del boosting de gradiente. No se submuestrean las variables: con
# solo dos, cada árbol se quedaría con una y no podría combinar carrera y sexo.
PARAMS_XGB = dict(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
                  colsample_bytree=1.0, missing=np.nan, random_state=42, n_jobs=-1)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def validar(variables, objetivo, datos):
    """Validación cruzada de 5 particiones. Devuelve el R² y el MAE de cada partición."""
    cv = cross_validate(xgb.XGBRegressor(**PARAMS_XGB), datos[variables].values, datos[objetivo].values,
                        cv=kf, scoring=('r2', 'neg_mean_absolute_error'))
    return cv['test_r2'], -cv['test_neg_mean_absolute_error']

# Modelo directo: carrera y sexo -> ciclismo
FEATURE_COLS = ['running_speed_kmh', 'sexo_M']
df_xgb = half.sample(min(len(half), 80000), random_state=42)[
    FEATURE_COLS + ['swim_speed_kmh', 'pace_cycling_kmh']].copy()
print(f"\nMuestra del modelo directo: {len(df_xgb):,} triatletas")

In [ ]:
# Modelo directo: validación cruzada y entrenamiento final
cv_r2, cv_mae = validar(FEATURE_COLS, 'pace_cycling_kmh', df_xgb)
print("Modelo directo (carrera -> ciclismo), validación cruzada de 5 particiones:")
print(f"  R²  = {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
print(f"  MAE = {cv_mae.mean():.2f} ± {cv_mae.std():.2f} km/h")

X = df_xgb[FEATURE_COLS].values
model_xgb = xgb.XGBRegressor(**PARAMS_XGB).fit(X, df_xgb['pace_cycling_kmh'].values)

In [ ]:
# Importancia de cada variable con valores SHAP (Figura 10 de la memoria)
NOMBRE_VAR = {'running_speed_kmh': 'Velocidad de carrera', 'pace_cycling_kmh': 'Velocidad en ciclismo',
              'sexo_M': 'Sexo (1 = hombre)'}
shap_values = shap.TreeExplainer(model_xgb).shap_values(X)
imp_shap = pd.Series(np.abs(shap_values).mean(axis=0),
                     index=[NOMBRE_VAR[f] for f in FEATURE_COLS]).sort_values()
print("Efecto medio de cada variable sobre la predicción (media de |SHAP|, km/h):")
print(imp_shap.sort_values(ascending=False).round(2).to_string())

fig, ax = plt.subplots(figsize=(7, 2.4))
ax.barh(imp_shap.index, imp_shap.values, color='#1f77b4')
for i, v in enumerate(imp_shap.values):
    ax.text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=9)
ax.set_xlabel('Efecto medio sobre la velocidad predicha (km/h)')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig('salidas/figuras/fig10_shap.png', dpi=200); plt.show()

In [ ]:
# Predicción de la velocidad en ciclismo de cada deportista (Tabla 13 y Figura 11).
# Entrada: su velocidad sostenida de carrera. La predicción se comunica con un
# margen conservador (x 0,85) porque el modelo aprende de velocidades de competición.
HISTORIAL_EVAL = 25   # sesiones de ciclismo necesarias para comparar con la velocidad real

filas = []
for letra, d in garmin_data.items():
    acts = d.get('activities', pd.DataFrame())
    run = acts[acts['sport'] == 'RUNNING'].dropna(subset=['avg_speed_kmh'])
    cyc = acts[acts['sport'] == 'CYCLING'].dropna(subset=['avg_speed_kmh'])
    if run.empty:
        continue
    x = {'running_speed_kmh': ritmo_sostenido(run),
         'sexo_M': 1.0 if sexo_por_atleta.get(letra, 'M') == 'M' else 0.0}
    pred = float(model_xgb.predict(np.array([[x[f] for f in FEATURE_COLS]]))[0])
    filas.append({'athlete': letra, 'sesiones_bici': len(cyc),
                  'carrera_p80': x['running_speed_kmh'], 'bici_real_p80': ritmo_sostenido(cyc),
                  'pred_kmh': pred, 'pred_conservador_kmh': pred * 0.85,
                  'tiempo_90km_min': 90 / (pred * 0.85) * 60})
pred_cyc = pd.DataFrame(filas).set_index('athlete')
pred_cyc['dif_sin_margen'] = pred_cyc['pred_kmh'] - pred_cyc['bici_real_p80']
pred_cyc['dif_con_margen'] = pred_cyc['pred_conservador_kmh'] - pred_cyc['bici_real_p80']
evaluables = pred_cyc[pred_cyc['sesiones_bici'] >= HISTORIAL_EVAL]
print(pred_cyc.round(1).to_string())
print(f"\nDeportistas con al menos {HISTORIAL_EVAL} sesiones de ciclismo: {', '.join(evaluables.index)}")
print(f"  Sin margen, la predicción supera la velocidad real entre "
      f"{evaluables['dif_sin_margen'].min():.1f} y {evaluables['dif_sin_margen'].max():.1f} km/h")
print(f"  Con margen, error medio absoluto: {evaluables['dif_con_margen'].abs().mean():.1f} km/h")

# Lista que usa el motor de planificación (sección 7)
predicciones_cyc = (pred_cyc.reset_index()[['athlete', 'pred_kmh', 'pred_conservador_kmh', 'tiempo_90km_min']]
                    .round(1).to_dict('records'))

# Figura 11. Velocidad real, predicha y comunicada
fig, ax = plt.subplots(figsize=(7, 3.8)); xs = np.arange(len(pred_cyc)); w = 0.26
series = [(pred_cyc['bici_real_p80'], '#7f7f7f', 'Real: velocidad sostenida en entrenamiento (p80)'),
          (pred_cyc['pred_kmh'], '#4C72B0', 'Predicción del modelo (XGBoost)'),
          (pred_cyc['pred_conservador_kmh'], '#55A868', 'Predicción comunicada (× 0,85)')]
for k, (vals, col, lab) in enumerate(series):
    barras = ax.bar(xs + (k - 1) * w, vals.values, w, color=col, label=lab)
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.4, f'{b.get_height():.1f}', ha='center', fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels([f'Atleta {l}' for l in pred_cyc.index]); ax.set_ylabel('km/h')
ax.set_ylim(0, max(36, pred_cyc['pred_kmh'].max() + 4))
ax.legend(fontsize=8, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig('salidas/figuras/fig11_ciclismo_pred_real.png', dpi=200); plt.show()

In [ ]:
# Modelo inverso: ciclismo y sexo -> carrera
FEAT_CYC2RUN = ['pace_cycling_kmh', 'sexo_M']
df_xgb_run = half.sample(min(len(half), 80000), random_state=7)[
    FEAT_CYC2RUN + ['swim_speed_kmh', 'running_speed_kmh']].copy()
cv_r2_r, cv_mae_r = validar(FEAT_CYC2RUN, 'running_speed_kmh', df_xgb_run)
print("Modelo inverso (ciclismo -> carrera), validación cruzada de 5 particiones:")
print(f"  R²  = {cv_r2_r.mean():.3f} ± {cv_r2_r.std():.3f}")
print(f"  MAE = {cv_mae_r.mean():.2f} ± {cv_mae_r.std():.2f} km/h")
model_xgb_run = xgb.XGBRegressor(**PARAMS_XGB).fit(df_xgb_run[FEAT_CYC2RUN].values,
                                                   df_xgb_run['running_speed_kmh'].values)

def apply_brick_factor(pace_kmh):
    """Tiempo de los 21,1 km con el factor de transición: 5 km al 90 %,
    10 km a la velocidad prevista y 6,1 km al 95 % (memoria, 3.4)."""
    v1, v2, v3 = pace_kmh * 0.90, pace_kmh, pace_kmh * 0.95
    return round((5 / v1 + 10 / v2 + 6.1 / v3) * 60, 1)

# Predicción para cada deportista con historial de ciclismo. Se comunica con
# un margen conservador de 0,5 km/h menos.
filas = []
for letra, d in garmin_data.items():
    acts = d.get('activities', pd.DataFrame())
    cyc = acts[acts['sport'] == 'CYCLING'].dropna(subset=['avg_speed_kmh'])
    run = acts[acts['sport'] == 'RUNNING'].dropna(subset=['avg_speed_kmh'])
    if len(cyc) < 2:
        continue
    x = {'pace_cycling_kmh': ritmo_sostenido(cyc),
         'sexo_M': 1.0 if sexo_por_atleta.get(letra, 'M') == 'M' else 0.0}
    pred = float(model_xgb_run.predict(np.array([[x[f] for f in FEAT_CYC2RUN]]))[0])
    pace = max(pred - 0.5, 5.0)
    filas.append({'athlete': letra, 'bici_p80': x['pace_cycling_kmh'], 'carrera_real_p80': ritmo_sostenido(run),
                  'pred_kmh': pred, 'pace_kmh': pace, 'tiempo_21km_min': apply_brick_factor(pace)})
pred_run = pd.DataFrame(filas).set_index('athlete')
pred_run['comunicada_menos_real'] = pred_run['pace_kmh'] - pred_run['carrera_real_p80']
print()
print(pred_run.round(1).to_string())
predicciones_run = pred_run.reset_index()[['athlete', 'pace_kmh', 'tiempo_21km_min']].round(1).to_dict('records')

### 6c. Comprobación: ¿cuánto aporta la natación?

Los modelos no usan la velocidad de natación: es el segmento que menos se relaciona con el tiempo final y
quien llega al triatlón desde la carrera o el ciclismo puede no tener ese dato. Esta celda mide lo que se
pierde al retirarla. Repite la validación añadiéndola, con las mismas filas, los mismos hiperparámetros y las
mismas particiones; solo cambia la lista de variables.

In [ ]:
comparacion = []
for nombre, datos, objetivo, base, r2_sin, mae_sin in [
        ('Directo (carrera -> ciclismo)', df_xgb, 'pace_cycling_kmh', FEATURE_COLS, cv_r2, cv_mae),
        ('Inverso (ciclismo -> carrera)', df_xgb_run, 'running_speed_kmh', FEAT_CYC2RUN, cv_r2_r, cv_mae_r)]:
    r2_con, mae_con = validar(base + ['swim_speed_kmh'], objetivo, datos)
    comparacion.append({'Modelo': nombre,
                        'R² sin natación': f'{r2_sin.mean():.3f} ± {r2_sin.std():.3f}',
                        'R² con natación': f'{r2_con.mean():.3f} ± {r2_con.std():.3f}',
                        'Diferencia de R²': round(r2_con.mean() - r2_sin.mean(), 3),
                        'MAE sin natación (km/h)': f'{mae_sin.mean():.2f}',
                        'MAE con natación (km/h)': f'{mae_con.mean():.2f}'})
comparacion = pd.DataFrame(comparacion).set_index('Modelo')
comparacion

,R² sin natación,R² con natación,Diferencia de R²,MAE sin natación (km/h),MAE con natación (km/h)
Modelo,,,,,
Directo (carrera -> ciclismo),0.467 ± 0.008,0.505 ± 0.006,0.04,2.38,2.29
Inverso (ciclismo -> carrera),0.457 ± 0.005,0.500 ± 0.004,0.04,1.25,1.20


**Hallazgo.** Añadir la natación sube el R² unas cuatro centésimas en los dos modelos (de 0,467 a 0,505 en el
directo y de 0,457 a 0,500 en el inverso), pero el error medio apenas cambia: de 2,38 a 2,28 km/h en ciclismo
y de 1,25 a 1,20 km/h en carrera, menos de un minuto sobre cada segmento. La mejora no compensa exigir un dato
que muchos deportistas no tienen.

### 6e. Sensibilidad a los hiperparámetros

Los hiperparámetros se fijaron a priori. Para comprobar que el resultado no depende de esa elección, se
repite la validación con una configuración más simple y otra más compleja, y se compara el R² de
entrenamiento con el de validación: si están cerca, el modelo no memoriza los datos.

In [ ]:
CONFIGURACIONES = {
    'Más simple (200 árboles, tasa 0,10, profundidad 3)': dict(n_estimators=200, learning_rate=0.10, max_depth=3),
    'Elegida (400 árboles, tasa 0,05, profundidad 5)':    dict(n_estimators=400, learning_rate=0.05, max_depth=5),
    'Más compleja (800 árboles, tasa 0,025, profundidad 7)': dict(n_estimators=800, learning_rate=0.025, max_depth=7),
}
filas = []
for etiqueta, datos, variables, objetivo in [
        ('Directo', df_xgb, FEATURE_COLS, 'pace_cycling_kmh'),
        ('Inverso', df_xgb_run, FEAT_CYC2RUN, 'running_speed_kmh')]:
    for nombre, cambios in CONFIGURACIONES.items():
        params = {**PARAMS_XGB, **cambios}
        cv = cross_validate(xgb.XGBRegressor(**params), datos[variables].values, datos[objetivo].values,
                            cv=kf, scoring=('r2', 'neg_mean_absolute_error'), return_train_score=True)
        filas.append({'Modelo': etiqueta, 'Configuración': nombre,
                      'R² validación': f"{cv['test_r2'].mean():.3f}",
                      'R² entrenamiento': f"{cv['train_r2'].mean():.3f}",
                      'MAE validación (km/h)': f"{-cv['test_neg_mean_absolute_error'].mean():.2f}"})
sensibilidad = pd.DataFrame(filas).set_index(['Modelo', 'Configuración'])
sensibilidad

**Hallazgo.** El resultado apenas depende de los hiperparámetros. En el modelo directo, el R² de validación
queda entre 0,466 y 0,468 en las tres configuraciones y el MAE es de 2,38 km/h en todas; en el inverso, entre
0,455 y 0,458 con un MAE de 1,25 km/h. El R² de entrenamiento de la configuración elegida (0,474 y 0,463) es
casi igual al de validación (0,467 y 0,457): el modelo no memoriza los datos. Con dos variables y 80.000
observaciones, el límite lo marca la información de las variables, no la complejidad del modelo.

### 6d. Resumen de resultados de la fase 1

Reúne las cifras de la validación (Tabla 12 de la memoria), la importancia de las variables, la comprobación
de la natación y las predicciones para cada deportista (Tabla 13).

In [ ]:
def minutos(dist, v):
    return dist / v * 60

v_bici, v_run = df_xgb['pace_cycling_kmh'].mean(), df_xgb_run['running_speed_kmh'].mean()
mae_b, mae_r = cv_mae.mean(), cv_mae_r.mean()
print("TABLA 12 — validación cruzada")
print(f"  Directo: R² {cv_r2.mean():.3f} ± {cv_r2.std():.3f} | MAE {mae_b:.2f} ± {cv_mae.std():.2f} km/h | "
      f"≈ {minutos(90, v_bici - mae_b) - minutos(90, v_bici):.0f} min en 90 km")
print(f"  Inverso: R² {cv_r2_r.mean():.3f} ± {cv_r2_r.std():.3f} | MAE {mae_r:.2f} ± {cv_mae_r.std():.2f} km/h | "
      f"{minutos(21.1, v_run) - minutos(21.1, v_run + mae_r):.0f}-"
      f"{minutos(21.1, v_run - mae_r) - minutos(21.1, v_run):.0f} min en 21,1 km")
print(f"  MAE en % de la velocidad media: ciclismo {mae_b / v_bici * 100:.1f} % (media {v_bici:.1f} km/h), "
      f"carrera {mae_r / v_run * 100:.1f} % (media {v_run:.1f} km/h)")
print("\nSHAP (Figura 10):", ', '.join(f'{k} {v:.2f} km/h' for k, v in imp_shap.sort_values(ascending=False).items()))
print("\nSENSIBILIDAD A LOS HIPERPARÁMETROS")
print(sensibilidad.to_string())
print("\nCOMPROBACIÓN DE LA NATACIÓN (apartado 5.2)")
print(comparacion[['R² sin natación', 'R² con natación', 'Diferencia de R²']].to_string())
print("\nTABLA 13 — predicciones frente a la velocidad real")
t13 = pd.DataFrame({
    'Ciclismo real (p80)': pred_cyc['bici_real_p80'].round(1),
    'Ciclismo predicho / comunicado': pred_cyc['pred_kmh'].round(1).astype(str) + ' / ' + pred_cyc['pred_conservador_kmh'].round(1).astype(str),
    'Tiempo previsto 90 km': pred_cyc['tiempo_90km_min'].apply(lambda m: f'{int(m // 60)} h {int(round(m % 60)):02d} min'),
    'Carrera real (p80)': pred_run['carrera_real_p80'].round(1),
    'Carrera comunicada': pred_run['pace_kmh'].round(1),
    'Tiempo previsto 21,1 km': pred_run['tiempo_21km_min'].apply(lambda m: f'{int(m // 60)} h {int(round(m % 60)):02d} min')})
print(t13.to_string())
print(f"\nCiclismo, deportistas con historial ({', '.join(evaluables.index)}): sin margen +{evaluables['dif_sin_margen'].min():.1f} a "
      f"+{evaluables['dif_sin_margen'].max():.1f} km/h; con margen "
      + ', '.join(f"{l} {v:+.1f}" for l, v in evaluables['dif_con_margen'].items())
      + f" km/h; error medio {evaluables['dif_con_margen'].abs().mean():.1f} km/h")
print(f"Carrera: la velocidad comunicada queda entre {pred_run['comunicada_menos_real'].min():+.1f} y "
      f"{pred_run['comunicada_menos_real'].max():+.1f} km/h respecto a la real")

**Hallazgos de la fase 1.**

- Con la velocidad de carrera y el sexo, el modelo directo explica casi la mitad de la variabilidad de la
  velocidad en ciclismo (R² = 0,467) con un error medio de 2,38 km/h, unos 15 minutos sobre los 90 km. El
  inverso alcanza un R² de 0,457 con un error de 1,25 km/h. La desviación entre particiones es muy pequeña:
  el resultado no depende de cómo se dividan los datos.
- La velocidad de carrera es la variable que más pesa (efecto medio de 2,19 km/h frente a 0,60 km/h del
  sexo), coherente con la idea de que lo que se transfiere es la capacidad aeróbica.
- Sin margen, el modelo directo sobreestima la velocidad real de entrenamiento de B, C y D entre 1,3 y
  9,2 km/h, porque aprende de velocidades de competición. Con el margen de 0,85 el error medio baja a
  3,3 km/h. Para A, que rueda a 10,3 km/h en desplazamientos urbanos, la predicción conservadora (23,0 km/h)
  sirve como estimación de partida.
- El modelo inverso subestima a los cuatro deportistas entre 1,1 y 3,9 km/h: aprende de carreras corridas al
  final de un 70.3, con la fatiga acumulada. Es un sesgo prudente, pero da objetivos lentos a buenos
  corredores.

## 7. Fase 2 — Motor de planificación semanal

Memoria, apartados 3.5 y 4.5. El plan se decide en capas:

1. **Tipo de semana** con las reglas en cascada de la Tabla 7 (TSB, Body Battery y estrés); la salud manda
   sobre la forma.
2. **Fase del macrociclo** a partir de la fecha de la prueba o de la duración mínima del plan según el nivel
   de experiencia (24 semanas para un deportista nuevo y 12 para uno experimentado).
3. **Zonas y ritmos** con el historial del deportista. El ritmo de rodaje se valida con carreras hechas en
   zona 2; si no las hay, se corrige con la FC de reserva, porque el ritmo medio de un corredor aficionado
   suele ser de zona 3 o 4.

En el cuaderno cada tipo de semana tiene sesiones fijas y las zonas se calculan con la FC de reserva. La
aplicación aplica las mismas reglas, pero con el TSB medio de siete días, zonas por porcentaje de la FC
máxima y sesiones generadas con las fórmulas de progresión del apartado 4.5.2 (ver sección 7c). Las Tablas 14
y 16 de la memoria salen de este motor.

In [ ]:
# --- Utilidades de formato: km/h -> ritmos legibles ---
def _pace_run(v_kmh):
    # km/h -> 'm:ss min/km'
    if pd.isna(v_kmh) or v_kmh <= 0: return 'N/D'
    m = 60.0 / v_kmh
    s = int(round((m - int(m)) * 60))
    if s == 60: m, s = m + 1, 0
    return f"{int(m)}:{s:02d} min/km"

def _pace_swim(v_kmh):
    # km/h -> 'm:ss por 100 m'
    if pd.isna(v_kmh) or v_kmh <= 0: return 'N/D'
    s = 360.0 / v_kmh
    return f"{int(s // 60)}:{int(s % 60):02d}/100 m"

# --- Macrociclo: ¿cuándo es la competencia? ---
# En la aplicación este valor lo introduce el deportista con un
# st.date_input ("¿Cuándo es tu 70.3?") o, si aún no tiene dorsal, con un
# selector de meses de entrenamiento. Aquí se configura por diccionario.
FECHA_COMPETENCIA = {}            # ej.: {'A': '2026-09-20'}
SEMANAS_PLAN_DEFAULT = 16         # macrociclo tipo si no hay fecha
SEMANA_ACTUAL_DEFAULT = 6         # semana del plan en la que está el atleta
CTL_OBJETIVO_70_3 = 70            # forma (CTL) para terminar un 70.3 con solvencia
PERFIL_ATLETA = {'A': 'nuevo'}     # 'nuevo' o 'experimentado' (en la aplicación lo elige el deportista)
SEMANAS_MINIMAS = {'nuevo': 24, 'experimentado': 12}   # 6 meses / 3 meses de plan mínimo
OBJETIVOS_70_3 = {}   # deportista experimentado con marca previa. Ejemplo (tiempos ficticios):
# OBJETIVOS_70_3['B'] = {'anterior': {'nat': '0:42', 'bici': '2:58',
#                                     'run': '2:05', 'total': '5:55'},
#                        'objetivo_total': '5:35'}

def calcular_fase(name):
    """Semanas hasta la competencia, fase del macrociclo y viabilidad.
    El perfil marca la duración mínima del plan: un atleta nuevo necesita
    al menos 24 semanas (6 meses); con experiencia, 12 (3 meses)."""
    perfil = PERFIL_ATLETA.get(name, 'experimentado')
    sem_min = SEMANAS_MINIMAS[perfil]
    f = FECHA_COMPETENCIA.get(name)
    if f:
        sem_rest = max(int(np.ceil((pd.Timestamp(f) - pd.Timestamp.today()).days / 7)), 0)
        sem_tot = sem_rest + SEMANA_ACTUAL_DEFAULT
    else:
        sem_tot = sem_min
        sem_rest = sem_tot - SEMANA_ACTUAL_DEFAULT
    viable = sem_tot >= sem_min
    aviso = ('' if viable else
             f"para un atleta {perfil} se recomiendan al menos {sem_min} semanas de "
             f"preparación y este plan solo tiene {sem_tot}; considera una carrera más "
             f"lejana o ajusta el objetivo (terminar bien, no competir)")
    if sem_rest <= 1:    fase = 'COMPETENCIA'
    elif sem_rest <= 3:  fase = 'TAPERING'
    elif sem_rest <= 6:  fase = 'PICO'
    elif sem_rest <= 12: fase = 'CONSTRUCCION'
    else:                fase = 'BASE'
    return {'fase': fase, 'semanas_restantes': sem_rest, 'semanas_totales': sem_tot,
            'perfil': perfil, 'semanas_minimas': sem_min,
            'plan_viable': viable, 'aviso': aviso,
            'fecha_competencia': f if f else
            f'sin fecha (plan de {sem_min} semanas por perfil {perfil})'}

def _hm2min(txt):
    """'h:mm' -> minutos."""
    p = [int(v) for v in str(txt).split(':')]
    return p[0] * 60 + p[1]

def _min2t(m):
    h, mm = divmod(int(round(m)), 60)
    return f"{h}:{mm:02d}"

def analizar_objetivo(name, sem_prep=12):
    """Perfil experimentado orientado a marca: de los tiempos del último
    70.3 y del objetivo se derivan los splits y ritmos objetivo por
    disciplina. El reparto de la mejora pondera dónde hay más margen en un
    70.3: bici 50%, carrera 35%, natación 15%. Viabilidad: ~0.5% de mejora
    por semana de preparación disponible."""
    obj = OBJETIVOS_70_3.get(name)
    if not obj or PERFIL_ATLETA.get(name, 'experimentado') == 'nuevo':
        return None
    ant = {k: _hm2min(v) for k, v in obj['anterior'].items()}
    t_obj = _hm2min(obj['objetivo_total'])
    transiciones = max(ant['total'] - (ant['nat'] + ant['bici'] + ant['run']), 4)
    mejora = ant['total'] - t_obj
    if mejora <= 0:
        return None
    reparto = {'nat': 0.15, 'bici': 0.50, 'run': 0.35}
    splits = {k: ant[k] - mejora * w for k, w in reparto.items()}
    mejora_pct = mejora / ant['total'] * 100
    return {
        'anterior_total': _min2t(ant['total']), 'objetivo_total': obj['objetivo_total'],
        'mejora_min': round(mejora, 1), 'mejora_pct': round(mejora_pct, 1),
        'viable': mejora_pct <= max(sem_prep, 1) * 0.5,
        'transiciones_min': round(transiciones, 1),
        'splits_obj': {k: _min2t(v) for k, v in splits.items()},
        'v_nat_obj': 1.9 / (splits['nat'] / 60),
        'v_bici_obj': 90 / (splits['bici'] / 60),
        'v_run_obj': 21.1 / (splits['run'] / 60),
    }

def calcular_zonas(name):
    """Zonas personalizadas. FC por Karvonen con la FC máxima observada y
    la FC en reposo reales; ritmos a partir de los últimos 90 días y de la
    predicción XGBoost de ciclismo. Si falta un dato se usa un valor por
    defecto razonable para un triatleta novato."""
    p = df_profiles.loc[name] if name in df_profiles.index else pd.Series(dtype=float)
    acts = df_act[df_act['athlete'] == name]

    # FC máxima: percentil 98 de las máximas registradas (filtra lecturas erróneas)
    fc_max = np.nan
    if not acts.empty and 'max_hr' in acts.columns:
        fc_max = acts['max_hr'].dropna().quantile(0.98)
    if pd.isna(fc_max): fc_max = 185.0
    fc_rep = p.get('resting_hr', np.nan)
    if pd.isna(fc_rep): fc_rep = 55.0
    fcr = fc_max - fc_rep                              # reserva cardiaca

    def _hr(lo, hi): return (int(fc_rep + fcr * lo), int(fc_rep + fcr * hi))
    hr = {'Z1': _hr(.50, .60), 'Z2': _hr(.60, .70), 'Z3': _hr(.70, .80),
          'Z4': _hr(.80, .90), 'Z5': _hr(.90, 1.0)}

    # Ritmo de carrera: la media reciente se toma como referencia de Z3.
    # La media del historial no es el ritmo de rodaje: en un deportista nuevo
    # suele corresponder a Z3-Z4. Por eso el ritmo de Z2 se valida con el
    # historial: si hay al menos tres carreras hechas con FC de Z2, se usa su
    # ritmo real; si no, se corrige con la fracción de la FC de reserva que
    # usa de media (Karvonen inverso), porque rodar en Z2 es más lento.
    v_run = p.get('pace_running_kmh', np.nan)
    if pd.isna(v_run): v_run = 9.2                     # ~6:31 min/km
    v_run_z2, fuente_z2 = np.nan, 'estimado (sin historial suficiente)'
    if not acts.empty and {'sport', 'avg_hr', 'avg_speed_kmh'} <= set(acts.columns):
        runs = acts[acts['sport'] == 'RUNNING'].dropna(subset=['avg_hr', 'avg_speed_kmh'])
        en_z2 = runs[runs['avg_hr'].between(hr['Z2'][0], hr['Z2'][1])]
        if len(en_z2) >= 3:
            v_run_z2 = en_z2['avg_speed_kmh'].median()
            fuente_z2 = f'validado con {len(en_z2)} carreras reales en Z2 de tu historial'
        elif not runs.empty:
            f_obs = (runs['avg_hr'].mean() - fc_rep) / fcr
            if f_obs > 0:
                corr = float(np.clip(0.65 / f_obs, 0.72, 1.0))
                v_run_z2 = v_run * corr
                fuente_z2 = (f'corregido por FC: tu historial va al {f_obs*100:.0f}% de '
                             f'reserva cardiaca (Z3-Z4), no a ritmo de rodaje')
    if pd.isna(v_run_z2):
        v_run_z2 = v_run * 0.93
    perfil = PERFIL_ATLETA.get(name, 'experimentado')
    if perfil == 'nuevo':                               # margen de prudencia
        v_run_z2 *= 0.95
        fuente_z2 += ' · margen -5% por perfil nuevo'
    # Velocidad de bici sostenible: predicción conservadora de la fase 1 (sección 6)
    v_bici = next((pr['pred_conservador_kmh'] for pr in predicciones_cyc
                   if pr['athlete'] == name), np.nan)
    if pd.isna(v_bici): v_bici = p.get('pace_cycling_kmh', np.nan)
    if pd.isna(v_bici): v_bici = 25.0
    v_nat = p.get('pace_swimming_kmh', np.nan)
    if pd.isna(v_nat): v_nat = 2.6                     # ~2:18/100 m

    return {'fc_max': int(fc_max), 'fc_rep': int(fc_rep), 'hr': hr,
            'fuente_z2': fuente_z2, 'perfil': perfil,
            'run_z2': v_run_z2, 'run_z3': v_run, 'run_z4': v_run * 1.10,
            'run_obj': v_run, 'bici_obj': v_bici, 'nat_obj': v_nat,
            'bici_z2': v_bici * 0.82, 'bici_z4': v_bici,
            'nat_z2': v_nat, 'nat_z4': v_nat * 1.08}


def construir_sesiones(plan_tipo, z, is_subject, fase='CONSTRUCCION', perfil='experimentado'):
    """Microciclo de 7 días con prescripción completa por sesión."""
    hr = z['hr']
    def fz(zo): return f"{hr[zo][0]}-{hr[zo][1]} ppm"
    def kmb(mins, v): return int(round(mins / 60 * v))      # km de bici
    def kmr(mins, v): return round(mins / 60 * v, 1)        # km de carrera
    ses = []
    def A(dia, dep, tipo, mins, km, det):
        ses.append({'dia': dia, 'deporte': dep, 'tipo': tipo,
                    'dur_min': mins, 'km': km, 'detalle': det})

    if plan_tipo == 'DESCANSO':
        A('Lunes a Domingo', 'Descanso', 'recuperación', 0, 0,
          f"Sin entrenamientos estructurados. Caminatas de 30-40 min si apetece, "
          f"siempre por debajo de {hr['Z1'][1]} ppm.")
        A('2-3 días de la semana', 'Movilidad', 'recuperación', 20, 0,
          "15-20 min de estiramientos y movilidad de cadera y hombros. "
          "Prioriza dormir 8 h o más: el descanso también es entrenamiento.")
        return ses

    if plan_tipo == 'REGENERATIVO':
        A('Martes', 'Natación', 'técnica', 30, 1.2,
          f"1.200 m muy suaves: 200 m de calentamiento; 8x50 m de técnica (punto "
          f"muerto, patada lateral) con 20 s de descanso; 400 m continuos a "
          f"{_pace_swim(z['nat_z2'])} o más lento; 200 m de vuelta a la calma.")
        A('Jueves', 'Ciclismo', 'rodaje suave', 60, kmb(60, z['bici_z2'] * 0.9),
          f"60 min en llano (desnivel < 150 m), unos {kmb(60, z['bici_z2'] * 0.9)} km a "
          f"{z['bici_z2'] * 0.9:.0f} km/h. FC en Z1-Z2 ({hr['Z1'][0]}-{hr['Z2'][1]} ppm), "
          f"cadencia alta (90-95 rpm), sin apretar en ningún momento.")
        A('Sábado', 'Carrera', 'trote regenerativo', 30, kmr(30, z['run_z2'] * 0.95),
          f"30 min de trote muy suave (~{kmr(30, z['run_z2'] * 0.95)} km) a "
          f"{_pace_run(z['run_z2'] * 0.95)} o más lento, FC por debajo de {hr['Z2'][0]} ppm. "
          f"Termina con 10 min de estiramientos.")
        return ses

    if plan_tipo == 'TAPER':
        A('Lunes', 'Descanso', 'recuperación', 0, 0,
          'Descanso completo. En el taper la forma ya está hecha: ahora se guarda.')
        A('Martes', 'Carrera', 'activación', 40, kmr(40, z['run_z2']),
          f"40 min (~{kmr(40, z['run_z2'])} km) en Z2 a {_pace_run(z['run_z2'])} con "
          f"4x90 s a ritmo de competencia ({_pace_run(z['run_obj'])}), recuperando 2 min al trote.")
        A('Miércoles', 'Natación', 'técnica', 35, 1.5,
          f"1.500 m suaves: 300 m de calentamiento; 6x50 m de técnica; 4x100 m a "
          f"{_pace_swim(z['nat_z2'])} con 20 s de descanso; 200 m de vuelta a la calma.")
        A('Jueves', 'Ciclismo', 'activación', 60, kmb(60, z['bici_z2']),
          f"60 min en llano: 3x3 min a ritmo de competencia ({z['bici_obj']:.0f} km/h, "
          f"FC máx {hr['Z3'][1]} ppm) con 5 min suaves entre bloques; el resto en Z1-Z2 "
          f"({hr['Z1'][0]}-{hr['Z2'][1]} ppm).")
        A('Viernes', 'Descanso', 'recuperación', 0, 0,
          'Descanso. Hidrátate bien y revisa el material de carrera.')
        A('Sábado', 'Ciclismo', 'activación corta', 40, kmb(40, z['bici_z2'])," ".join((
          f"40 min suaves con 3 aceleraciones de 1 min, y 10 min de trote al bajar.",
          "Piernas despiertas, no cansadas.")))
        A('Domingo', 'Carrera', 'rodaje corto', 35, kmr(35, z['run_z2']),
          f"35 min en Z2 a {_pace_run(z['run_z2'])}. Volumen semanal ~50% del habitual; "
          f"la intensidad, solo en toques cortos.")
        return ses

    if plan_tipo == 'COMPETENCIA':
        t_bici = 90 / z['bici_obj'] * 60
        A('Lunes', 'Descanso', 'recuperación', 0, 0, 'Descanso o 20 min de movilidad suave.')
        A('Martes', 'Natación', 'activación', 25, 1.0,
          f"1.000 m suaves con 4x50 m a ritmo de prueba ({_pace_swim(z['nat_z2'])}).")
        A('Miércoles', 'Carrera', 'activación', 30, kmr(30, z['run_z2']),
          f"30 min en Z2 con 4x60 s a {_pace_run(z['run_obj'])}.")
        A('Jueves', 'Ciclismo', 'activación', 40, kmb(40, z['bici_z2']),
          '40 min suaves con 3x90 s a ritmo de carrera. Revisión completa de la bici.')
        A('Viernes', 'Descanso', 'recuperación', 0, 0,
          'Descanso total. Prepara material, dorsales y nutrición de carrera.')
        A('Sábado', 'Ciclismo', 'activación pre-carrera', 25, kmb(15, z['bici_z2']),
          '15 min de bici suave + 8 min de trote con 3 rectas. Cena conocida y dormir temprano.')
        A('Domingo', 'Competencia', '70.3', 0, 113,
          f"Día de carrera. Natación 1,9 km a {_pace_swim(z['nat_z2'])} sin esprintar la salida; "
          f"bici 90 km a {z['bici_obj']:.0f} km/h (~{int(t_bici // 60)}h {int(t_bici % 60):02d}, "
          f"FC {hr['Z3'][0]}-{hr['Z3'][1]} ppm, come cada 30-40 min); carrera 21,1 km saliendo a "
          f"{_pace_run(z['run_z2'])} los primeros 5 km (brick effect) y cerrando a "
          f"{_pace_run(z['run_obj'])} si las sensaciones acompañan.")
        return ses

    if plan_tipo == 'MANTENIMIENTO':
        A('Lunes', 'Descanso', 'recuperación', 0, 0,
          'Descanso completo o 20 min de movilidad suave.')
        A('Martes', 'Natación', 'técnica + aeróbico', 45, 1.6,
          f"1.600 m: 300 m de calentamiento; 6x50 m de técnica con 15 s de descanso; "
          f"8x100 m a {_pace_swim(z['nat_z2'])} con 20 s de descanso entre cada 100; "
          f"200 m suaves de vuelta a la calma.")
        A('Miércoles', 'Ciclismo', 'rodaje Z2', 90, kmb(90, z['bici_z2']),
          f"90 min en terreno llano o rodillo (desnivel < 300 m), unos "
          f"{kmb(90, z['bici_z2'])} km a {z['bici_z2']:.0f} km/h de media. FC en Z2 "
          f"({fz('Z2')}), cadencia 85-95 rpm. Si la FC pasa de {hr['Z2'][1]} ppm, "
          f"baja el ritmo: el objetivo es construir base aeróbica, no ir rápido.")
        A('Jueves', 'Carrera', 'rodaje Z2 + rectas', 50, kmr(50, z['run_z2']),
          f"50 min (~{kmr(50, z['run_z2'])} km) a {_pace_run(z['run_z2'])}, FC en Z2 "
          f"({fz('Z2')}). Termina con 4 rectas de 80 m progresivas (acelerando sin "
          f"llegar a esprintar), recuperando al trote entre ellas.")
        A('Viernes', 'Natación', 'continuo', 50, 2.0,
          f"2.000 m continuos a {_pace_swim(z['nat_z2'])}, respirando cada 3 brazadas. "
          f"Cada 400 m intercala 50 m de técnica para resetear la postura.")
        A('Sábado', 'Ciclismo', 'salida larga', 135, kmb(135, z['bici_z2']),
          f"2 h 15 min (~{kmb(135, z['bici_z2'])} km) con 400-600 m de desnivel "
          f"acumulado. FC en Z2 ({fz('Z2')}); en las subidas, sentado y sin pasar de "
          f"{hr['Z3'][0]} ppm. Come algo cada 45 min: la nutrición también se entrena.")
        A('Sábado (al bajar de la bici)', 'Carrera', 'brick', 15, kmr(15, z['run_z2']),
          f"Transición: 15 min de trote (~{kmr(15, z['run_z2'])} km) a "
          f"{_pace_run(z['run_z2'])} nada más bajar de la bici. Las piernas pesarán "
          f"los primeros 5 min: es normal y es exactamente lo que pasa en el 70.3.")
        A('Domingo', 'Carrera', 'tirada larga progresiva', 70, kmr(70, z['run_z2']),
          f"70 min (~{kmr(70, z['run_z2'])} km): 55 min en Z2 a {_pace_run(z['run_z2'])} "
          f"y los últimos 15 min en Z3 ({fz('Z3')}) a {_pace_run(z['run_z3'])}.")

    if plan_tipo == 'CARGA':
        reps = 4 if perfil == 'nuevo' else 6        # menos repeticiones para el nuevo
        bloq = 3 if perfil == 'nuevo' else 4
        km_series = round(kmr(25, z['run_z2']) + reps * 1.2, 1)
        A('Lunes', 'Descanso', 'recuperación', 0, 0,
          'Descanso completo: la semana es exigente y empieza con series.')
        if fase == 'BASE':
            A('Martes', 'Carrera', 'fartlek Z3', 50, kmr(50, z['run_z2']),
              f"50 min (~{kmr(50, z['run_z2'])} km): 15 min en Z2; 8x1 min a "
              f"{_pace_run(z['run_z3'])} (FC máx {hr['Z3'][1]} ppm) con 2 min suaves "
              f"entre cambios; 10 min de vuelta a la calma. Aún sin Zona 4: en fase "
              f"de base primero se construyen los cimientos.")
        else:
            A('Martes', 'Carrera', 'series Z4', 55, km_series,
              f"~{km_series} km en total: 15 min de calentamiento en Z2; {reps}x800 m a "
              f"{_pace_run(z['run_z4'])} llegando a Z4 ({fz('Z4')}) al final de cada "
              f"repetición, recuperando 400 m al trote (2-3 min); 10 min suaves para cerrar.")
        A('Miércoles', 'Natación', 'series', 55, 2.2,
          f"2.200 m: 400 m de calentamiento; 10x100 m a {_pace_swim(z['nat_z4'])} con "
          f"30 s de descanso; 200 m de técnica; 200 m suaves. El objetivo es sostener "
          f"el mismo ritmo del primer al último 100.")
        if fase == 'BASE':
            A('Jueves', 'Ciclismo', 'fuerza en baja cadencia', 75, kmb(75, z['bici_z2']),
              f"75 min: 15 min de calentamiento; 5x5 min a 60-65 rpm en Z3 ({fz('Z3')}) "
              f"sentado (fuerza específica sin impacto), recuperando 3 min a 90 rpm; "
              f"15 min suaves.")
        else:
            A('Jueves', 'Ciclismo', 'potencia Z4', 80, kmb(80, z['bici_z2'] * 1.05),
              f"80 min: 15 min de calentamiento; {bloq}x8 min a {z['bici_z4']:.0f} km/h en llano "
              f"(FC en Z4, {fz('Z4')}; con potenciómetro, 90-100 % de tu FTP), cadencia "
              f"80-85 rpm, recuperando 4 min de pedaleo suave entre bloques; 15 min de "
              f"vuelta a la calma.")
        A('Viernes', 'Natación', 'continuo', 55, 2.4,
          f"2.400 m continuos a {_pace_swim(z['nat_z2'])} + 6x50 m de técnica al final.")
        A('Sábado', 'Ciclismo', 'salida larga con desnivel', 165, kmb(165, z['bici_z2']),
          f"2 h 45 min (~{kmb(165, z['bici_z2'])} km) con 600-900 m de desnivel. FC en "
          f"Z2-Z3 ({hr['Z2'][0]}-{hr['Z3'][1]} ppm); las subidas largas, sentado a "
          f"70-75 rpm, simulando el esfuerzo del día de la carrera. Practica comer y "
          f"beber sobre la bici."
          + (f" Últimos 40 min a ritmo de competencia ({z['bici_obj']:.0f} km/h)."
             if fase == 'PICO' else ""))
        A('Sábado (al bajar de la bici)', 'Carrera', 'brick', 20, kmr(20, z['run_z2']),
          f"20 min de trote (~{kmr(20, z['run_z2'])} km) a {_pace_run(z['run_z2'])} nada "
          f"más guardar la bici, controlando el ritmo aunque las piernas no respondan.")
        A('Domingo', 'Carrera', 'tirada larga con ritmo', 90, kmr(90, z['run_z2']),
          f"90 min (~{kmr(90, z['run_z2'])} km) en Z2 con 3 bloques de 10 min a tu ritmo "
          f"objetivo de 70.3 ({_pace_run(z['run_obj'])}, FC máx {hr['Z3'][1]} ppm), "
          f"recuperando 5 min suaves entre bloques.")

    # Sesión extra de bici para el caso de evaluación (su disciplina débil),
    # insertada en orden cronológico (tras la sesión del viernes)
    if is_subject and plan_tipo in ('MANTENIMIENTO', 'CARGA'):
        idx = max(i for i, s in enumerate(ses) if s['dia'].startswith('Viernes')) + 1
        extra = {'dia': 'Viernes (sesión extra)', 'deporte': 'Ciclismo',
                 'tipo': 'rodillo cadencia', 'dur_min': 45,
                 'km': kmb(45, z['bici_z2'] * 0.9),
                 'detalle': (
                     f"45 min de rodillo o llano: 10 min suaves; 6x(2 min a 100-105 rpm "
                     f"+ 2 min a 90 rpm) en Z2 ({fz('Z2')}); 10 min suaves. Trabajo de "
                     f"cadencia para ganar eficiencia de pedaleo, tu punto de mayor "
                     f"margen de mejora.")}
        ses.insert(idx, extra)
    return ses


def generar_plan_semanal(name):
    fd  = fitness_data.get(name, pd.DataFrame())
    d   = garmin_data.get(name, {})
    uds = d.get('uds', pd.DataFrame())
    rd  = d.get('readiness', pd.DataFrame())

    CTL = fd['CTL'].iloc[-1] if not fd.empty else np.nan
    ATL = fd['ATL'].iloc[-1] if not fd.empty else np.nan
    TSB = fd['TSB'].iloc[-1] if not fd.empty else np.nan
    bb  = uds['body_battery'].dropna().iloc[-1] if not uds.empty and not uds['body_battery'].dropna().empty else np.nan
    hrv = rd['hrv_weekly_avg'].dropna().iloc[-1] if not rd.empty and not rd['hrv_weekly_avg'].dropna().empty else np.nan
    stress = uds['stress_avg'].dropna().mean() if not uds.empty else np.nan
    is_subject = (name == SUJETO_PRUEBA)

    # Reglas de decisión (cascada TSB -> Body Battery -> estrés)
    if not pd.isna(TSB) and TSB < -25:
        plan_tipo = 'DESCANSO'
    elif not pd.isna(bb) and bb < 20:
        plan_tipo = 'REGENERATIVO'
    elif not pd.isna(TSB) and TSB < -10:
        plan_tipo = 'MANTENIMIENTO'
    elif not pd.isna(TSB) and TSB > 10 and (pd.isna(stress) or stress < 70):
        plan_tipo = 'CARGA'
    else:
        plan_tipo = 'MANTENIMIENTO'

    # Macrociclo: la fase modula el tipo de semana. La salud manda:
    # DESCANSO y REGENERATIVO nunca se anulan.
    per = calcular_fase(name)
    if plan_tipo in ('MANTENIMIENTO', 'CARGA'):
        if per['fase'] == 'COMPETENCIA':
            plan_tipo = 'COMPETENCIA'
        elif per['fase'] == 'TAPERING':
            plan_tipo = 'TAPER'

    zonas = calcular_zonas(name)
    # Perfil experimentado con objetivo de marca: los ritmos objetivo de
    # las sesiones de calidad y del plan de carrera salen de sus splits
    objetivo = analizar_objetivo(name, per['semanas_totales'] - SEMANA_ACTUAL_DEFAULT)
    if objetivo:
        zonas['run_obj']  = objetivo['v_run_obj']
        zonas['bici_obj'] = objetivo['v_bici_obj']
        zonas['nat_obj']  = objetivo['v_nat_obj']
    sesiones = construir_sesiones(plan_tipo, zonas, is_subject, per['fase'], per['perfil'])

    # Evaluación de progresión: rampa de CTL necesaria para llegar al objetivo
    if pd.isna(CTL):
        ramp, progreso = np.nan, 'N/D'
    elif per['fase'] in ('TAPERING', 'COMPETENCIA'):
        # En la fase final ya no se construye forma: la rampa no aplica
        ramp, progreso = 0.0, 'fase final: mantener, no construir'
    else:
        ramp = (CTL_OBJETIVO_70_3 - CTL) / max(per['semanas_restantes'], 1)
        if ramp <= 0:    progreso = 'objetivo de forma alcanzado'
        elif ramp <= 3:  progreso = 'en ruta'
        elif ramp <= 6:  progreso = 'justo: evitar semanas perdidas'
        else:            progreso = 'riesgo: forma lejos del objetivo'

    tot = {'Natación': 0, 'Ciclismo': 0, 'Carrera': 0}
    km  = {'Natación': 0.0, 'Ciclismo': 0.0, 'Carrera': 0.0}
    for s in sesiones:
        if s['deporte'] in tot:
            tot[s['deporte']] += s['dur_min']
            km[s['deporte']]  += s['km']

    return {
        'athlete': name, 'CTL': CTL, 'ATL': ATL, 'TSB': TSB,
        'body_battery': bb, 'hrv': hrv, 'stress': stress,
        'plan_tipo': plan_tipo,
        'fase': per['fase'], 'semanas_restantes': per['semanas_restantes'],
        'fecha_competencia': per['fecha_competencia'],
        'perfil': per['perfil'], 'plan_viable': per['plan_viable'], 'aviso': per['aviso'],
        'ctl_objetivo': CTL_OBJETIVO_70_3,
        'ramp_ctl_semana': round(float(ramp), 1) if not pd.isna(ramp) else np.nan,
        'progreso': progreso,
        'natacion_min': tot['Natación'], 'ciclismo_min': tot['Ciclismo'],
        'carrera_min': tot['Carrera'],
        'natacion_km': round(km['Natación'], 1), 'ciclismo_km': round(km['Ciclismo']),
        'carrera_km': round(km['Carrera'], 1),
        'sesiones_semana': sum(1 for s in sesiones if s['deporte'] in tot),
        'sujeto_prueba': is_subject,
        'zonas': zonas, 'sesiones': sesiones, 'objetivo': objetivo,
    }

planes = [generar_plan_semanal(n) for n in garmin_data]

# df_planes: solo columnas escalares (las anidadas van a df_sesiones)
df_planes = pd.DataFrame([{k: v for k, v in p.items()
                           if k not in ('zonas', 'sesiones', 'objetivo')} for p in planes])

# Tabla plana de sesiones (una fila por sesión) para persistencia y dashboard
df_sesiones = pd.DataFrame([{'athlete': p['athlete'], 'plan_tipo': p['plan_tipo'], **s}
                            for p in planes for s in p['sesiones']])

print("Planes semanales (resumen):\n")
print(df_planes[['athlete', 'TSB', 'fase', 'semanas_restantes', 'plan_tipo',
                 'progreso', 'sesiones_semana', 'natacion_km', 'ciclismo_km',
                 'carrera_km']].to_string(index=False))
print(f"\nSesiones individuales generadas: {len(df_sesiones)}")

In [ ]:
# Tabla 14 de la memoria: estado de carga y plan generado para cada deportista
NOMBRE_FASE = {'BASE': 'Base', 'CONSTRUCCION': 'Construcción', 'PICO': 'Pico',
               'TAPERING': 'Tapering', 'COMPETENCIA': 'Competición'}
NOMBRE_TIPO = {'DESCANSO': 'Descanso', 'REGENERATIVO': 'Regenerativa', 'MANTENIMIENTO': 'Mantenimiento',
               'CARGA': 'Carga', 'TAPER': 'Taper', 'COMPETENCIA': 'Competición'}
tabla_14 = pd.DataFrame([{
    'Atleta': p['athlete'], 'CTL': round(p['CTL'], 1), 'ATL': round(p['ATL'], 1), 'TSB': round(p['TSB'], 1),
    'Fase': f"{NOMBRE_FASE[p['fase']]} ({p['semanas_restantes']} semanas)",
    'Tipo de semana': NOMBRE_TIPO[p['plan_tipo']], 'Sesiones': p['sesiones_semana'],
    'Natación / ciclismo / carrera (km)': f"{p['natacion_km']} / {p['ciclismo_km']} / {p['carrera_km']}"}
    for p in planes]).set_index('Atleta')
tabla_14

### 7b. Simulación retrospectiva de las reglas (Tabla 16)

Memoria, apartado 5.3. Las reglas de la Tabla 7 se aplican a las últimas 16 semanas de cada deportista,
evaluando cada lunes el estado del día anterior, igual que haría el motor. Se comparan tres versiones:

- **Reglas actuales:** TSB del día anterior.
- **TSB medio de 7 días:** misma regla con la media de la última semana. Es la que usa la aplicación.
- **Umbrales proporcionales a la CTL:** −25, −10 y +10 pasan a −0,36, −0,14 y +0,14 veces la CTL, con el TSB
  medio de 7 días. Es solo un análisis de sensibilidad.

El Body Battery es el último valor disponible y el estrés, la media del historial hasta ese día, como en el
motor.

In [ ]:
def tipo_semana(tsb, bb, estres, ctl, proporcional=False):
    """Reglas de la Tabla 7. Devuelve 0 descanso, 1 regenerativa, 2 mantenimiento, 3 carga."""
    u1, u2, u3 = (-0.36 * ctl, -0.14 * ctl, 0.14 * ctl) if proporcional else (-25, -10, 10)
    if not pd.isna(tsb) and tsb < u1: return 0
    if not pd.isna(bb) and bb < 20: return 1
    if not pd.isna(tsb) and tsb < u2: return 2
    if not pd.isna(tsb) and tsb > u3 and (pd.isna(estres) or estres < 70): return 3
    return 2

def simular(letra, semanas=16):
    fd = fitness_data[letra].set_index('date')
    uds = garmin_data[letra].get('uds', pd.DataFrame())
    uds = uds.set_index('date') if not uds.empty else pd.DataFrame()
    fin = fd.index.max()
    lunes = pd.date_range(fin - pd.Timedelta(days=7 * semanas), fin, freq='W-MON')[-semanas:]
    cuenta = {v: [0, 0, 0, 0] for v in ('dia', 'media7', 'proporcional')}
    for l in lunes:
        d = l - pd.Timedelta(days=1)
        if d not in fd.index:
            continue
        tsb, ctl = fd.loc[d, 'TSB'], fd.loc[d, 'CTL']
        tsb7 = fd.loc[d - pd.Timedelta(days=6):d, 'TSB'].mean()
        bb = estres = np.nan
        if not uds.empty:
            serie_bb = uds.loc[:d, 'body_battery'].dropna()
            bb = serie_bb.iloc[-1] if not serie_bb.empty else np.nan
            estres = uds.loc[:d, 'stress_avg'].mean()
        cuenta['dia'][tipo_semana(tsb, bb, estres, ctl)] += 1
        cuenta['media7'][tipo_semana(tsb7, bb, estres, ctl)] += 1
        cuenta['proporcional'][tipo_semana(tsb7, bb, estres, ctl, proporcional=True)] += 1
    ultimo_anio = fitness_data[letra].tail(365)
    fmt = lambda c: ' / '.join(map(str, c))
    return {'Atleta': letra,
            'Reglas actuales (TSB del día anterior)': fmt(cuenta['dia']),
            'Variante: TSB medio de 7 días': fmt(cuenta['media7']),
            'Variante: umbrales proporcionales a la CTL': fmt(cuenta['proporcional']),
            'Días con TSB < −10 en el último año (%)': round((ultimo_anio['TSB'] < -10).mean() * 100, 1)}

tabla_16 = pd.DataFrame([simular(l) for l in fitness_data]).set_index('Atleta')
print("Semanas de descanso / regenerativa / mantenimiento / carga en las últimas 16 semanas")
tabla_16

**Hallazgo.** Con el TSB de un solo día, el motor prescribe descanso entre cuatro y siete semanas de cada
dieciséis en A, B y C, demasiado a menudo: su TSB diario oscila con brusquedad y un umbral de −25 se cruza con
facilidad. Con la media de siete días, las semanas de descanso bajan a entre cero y dos sin cambiar los
umbrales; los umbrales proporcionales a la CTL no mejoran más. El problema es la volatilidad del TSB diario,
y por eso la aplicación usa la media de siete días.

### 7c. Progresión del plan con las fórmulas de la aplicación (Tabla 15)

Memoria, apartado 4.5.2. La aplicación calcula el volumen relativo de cada semana como el producto de la
rampa, el factor de fase, el nivel y la descarga, y escala con él las duraciones de referencia de cada sesión.
Esta celda reproduce esas fórmulas para un deportista experimentado con un plan de 12 semanas, semanas de
tipo carga y una velocidad de zona 2 en bicicleta de 24 km/h. Los ritmos mejoran un 5 % a lo largo del plan
(un 10 % para un deportista nuevo).

In [ ]:
FACTOR_FASE = {'BASE': 0.80, 'CONSTRUCCION': 1.0, 'PICO': 1.0, 'TAPERING': 0.55, 'COMPETENCIA': 0.35}

def fase_de(semanas_restantes):
    if semanas_restantes <= 1: return 'COMPETENCIA'
    if semanas_restantes <= 3: return 'TAPERING'
    if semanas_restantes <= 6: return 'PICO'
    if semanas_restantes <= 12: return 'CONSTRUCCION'
    return 'BASE'

def progresion(semanas=12, nivel='experimentado', v_bici_z2=24.0):
    filas = []
    for s in range(1, semanas + 1):
        fase = fase_de(semanas - s + 1)          # las semanas restantes incluyen la actual
        prog = s / semanas
        rampa = 0.45 + 0.55 * min(prog / 0.85, 1.0)
        descarga = s % 4 == 0 and fase in ('BASE', 'CONSTRUCCION', 'PICO') and s < semanas - 2
        vol = rampa * FACTOR_FASE[fase] * (0.80 if nivel == 'nuevo' else 1.0) * (0.6 if descarga else 1.0)
        fondo_min = max(int(round(220 * vol / 5.0) * 5), 20)
        v = v_bici_z2 * (1 + (0.10 if nivel == 'nuevo' else 0.05) * prog)
        if fase == 'COMPETENCIA': tipo = 'Semana de la prueba'
        elif descarga: tipo = 'Descarga'
        elif fase in ('BASE', 'CONSTRUCCION', 'PICO') and prog >= 0.25: tipo = 'Brick'
        else: tipo = 'Sin brick'
        filas.append({'Semana': s, 'Fase': NOMBRE_FASE[fase], 'Volumen relativo': round(vol, 2),
                      'Fondo de bicicleta': f'{fondo_min / 60 * v:.0f} km', 'Tipo': tipo})
    return pd.DataFrame(filas).set_index('Semana')

tabla_15 = progresion(12, 'experimentado')
tabla_15.loc[4:]

## 8. Capa explicativa (Gemini o plantillas)

Memoria, apartado 4.6. El modelo de lenguaje no decide nada: recibe el plan ya calculado (sesiones con km,
ritmos, pulsaciones, series y desniveles) y solo lo redacta en lenguaje sencillo, con la instrucción de
conservar todas las cifras. Sin clave de Gemini se usan plantillas, que también incluyen la prescripción
completa. Los resultados de la memoria se generaron en modo plantilla.

In [ ]:
# Cabeceras por tipo de semana: explican el porqué fisiológico
CABECERAS = {
    'DESCANSO': ("Semana de descanso. Tu TSB de {tsb} indica una sobrecarga "
                 "importante: el cuerpo necesita absorber el entrenamiento "
                 "acumulado. Esta semana no hay sesiones de calidad; caminar, "
                 "estirar y dormir bien también es entrenar."),
    'REGENERATIVO': ("Semana de regeneración: tu Body Battery está en {bb}, "
                     "señal de poca energía disponible. Bajamos el volumen a "
                     "unas {horas} horas, todo en Zona 1-2 ({z12}): tienes que poder "
                     "hablar con comodidad en cada sesión."),
    'MANTENIMIENTO': ("Semana de mantenimiento (TSB {tsb}): unas {horas} horas "
                      "de trabajo aeróbico de base en Zona 2 ({z2}), los "
                      "cimientos del 70.3. Nada de heroicidades: la constancia "
                      "es la que gana esta carrera."),
    'TAPER': ("Semana de tapering: a {sem} semanas del 70.3 reducimos el "
              "volumen a la mitad manteniendo toques cortos de intensidad. La "
              "fatiga se va, la forma se queda: vas a llegar fresco."),
    'COMPETENCIA': ("Semana de competencia. Todo lo que había que entrenar ya "
                    "está entrenado: días cortos, activaciones y el domingo tu "
                    "70.3, con el pacing calculado a partir de tus propios datos."),
    'CARGA': ("Semana de carga: estás asimilando bien (TSB {tsb}) y toca "
              "apretar. Unas {horas} horas con dos sesiones de calidad: series "
              "en carrera y potencia en bici, ambas en Zona 4 ({z4}). El resto "
              "de la semana se queda en Zona 2 ({z2})."),
}

def _zonas_resumen(plan):
    z = plan['zonas']
    return (f"Tus referencias de esta semana — FC máx: {z['fc_max']} ppm | "
            f"FC reposo: {z['fc_rep']} ppm | "
            f"Z2: {z['hr']['Z2'][0]}-{z['hr']['Z2'][1]} ppm | "
            f"Z4: {z['hr']['Z4'][0]}-{z['hr']['Z4'][1]} ppm | "
            f"Rodaje: {_pace_run(z['run_z2'])} | Series: {_pace_run(z['run_z4'])} | "
            f"Bici Z2: {z['bici_z2']:.0f} km/h | Natación: {_pace_swim(z['nat_z2'])}")

def _sesiones_texto(plan):
    return "\n".join(f"  - {s['dia']} | {s['deporte']} ({s['tipo']}): {s['detalle']}"
                     for s in plan['sesiones'])

def generar_explicacion(plan):
    z = plan['zonas']
    cabecera = CABECERAS.get(plan['plan_tipo'], CABECERAS['MANTENIMIENTO']).format(
        tsb=f"{plan['TSB']:.1f}" if not pd.isna(plan['TSB']) else 'N/D',
        bb=f"{plan['body_battery']:.0f}" if not pd.isna(plan['body_battery']) else 'N/D',
        horas=f"{(plan['natacion_min'] + plan['ciclismo_min'] + plan['carrera_min']) / 60:.0f}",
        z12=f"{z['hr']['Z1'][0]}-{z['hr']['Z2'][1]} ppm",
        z2=f"{z['hr']['Z2'][0]}-{z['hr']['Z2'][1]} ppm",
        z4=f"{z['hr']['Z4'][0]}-{z['hr']['Z4'][1]} ppm",
        sem=plan.get('semanas_restantes', 'N/D'))

    ctl_txt = f"{plan['CTL']:.0f}" if not pd.isna(plan['CTL']) else 'N/D'
    linea_fase = (f"Macrociclo: fase {plan['fase'].capitalize()} · faltan "
                  f"{plan['semanas_restantes']} semanas para el 70.3 "
                  f"({plan['fecha_competencia']}) · progresión de forma: "
                  f"{plan['progreso']} (CTL {ctl_txt} / objetivo {plan['ctl_objetivo']}).")

    if not plan.get('plan_viable', True):
        linea_fase += "\nAVISO IMPORTANTE: " + plan['aviso'] + "."
    linea_val = f"Validación de ritmos: {z.get('fuente_z2', 'N/D')}."
    if plan.get('objetivo'):
        o = plan['objetivo']
        linea_val += (f"\nObjetivo de marca: {o['anterior_total']} → {o['objetivo_total']} "
                      f"(-{o['mejora_min']:.0f} min, {o['mejora_pct']}%). Splits objetivo: "
                      f"natación {o['splits_obj']['nat']} · bici {o['splits_obj']['bici']} · "
                      f"carrera {o['splits_obj']['run']} · transiciones {_min2t(o['transiciones_min'])}."
                      + ("" if o['viable'] else
                         " AVISO: mejora ambiciosa para las semanas disponibles; el plan prioriza la bici."))

    texto = (linea_fase + "\n\n" + cabecera + "\n\n" + _zonas_resumen(plan)
             + "\n" + linea_val
             + "\n\nTu semana, día a día:\n" + _sesiones_texto(plan))

    # Semana de competencia: se añade el plan de carrera km a km
    if plan['plan_tipo'] == 'COMPETENCIA' and 'texto_plan_competencia' in globals():
        texto += "\n\nPLAN DE COMPETENCIA, KM A KM:" + texto_plan_competencia(plan)
    # Cuadro final «Para entender tu plan» (glosario)
    if 'bloque_glosario' in globals():
        texto += "\n\n" + bloque_glosario(plan)
    if plan['sujeto_prueba']:
        texto += ("\n\nTu base de carrera y natación es tu mayor activo para "
                  "el 70.3. El ciclismo es tu área de crecimiento; por eso tu semana "
                  "lleva una sesión extra de bici. Cada hora sobre la bici suma.")

    if USE_LLM:
        prompt = (
            "Eres entrenador de triatlón. Reescribe el siguiente plan semanal en un "
            "lenguaje cercano y motivador para un atleta novato, en español "
            "(máximo 3 párrafos de introducción + la lista de sesiones día a día).\n"
            f"Estado: CTL={plan['CTL']:.1f}, ATL={plan['ATL']:.1f}, TSB={plan['TSB']:.1f}, "
            f"BodyBattery={plan['body_battery']}, HRV={plan['hrv']}. "
            f"Tipo de semana: {plan['plan_tipo']}. Fase del macrociclo: {plan['fase']} "
            f"(faltan {plan['semanas_restantes']} semanas; progresión: {plan['progreso']}).\n"
            f"Referencias: {_zonas_resumen(plan)}\n"
            f"Sesiones:\n{_sesiones_texto(plan)}\n"
            + ("Contexto: el deportista prepara su primer 70.3; tiene base en carrera y "
               "natación pero poca experiencia en ciclismo.\n" if plan['sujeto_prueba'] else "")
            + "Reglas: conserva EXACTAMENTE todas las cifras (km, ritmos, pulsaciones, "
              "series, recuperaciones, desniveles, cadencias); explica el porqué "
              "fisiológico de las dos decisiones más importantes de la semana en "
              "lenguaje simple; cierra con motivación realista hacia el 70.3.")
        try:
            return llm_model.generate_content(prompt).text
        except Exception as e:
            return f"[Error del LLM: {e}]\n\n{texto}"
    return texto

for plan in planes:
    print("=" * 70)
    print(f"Atleta {plan['athlete']}{' (caso de evaluación)' if plan['sujeto_prueba'] else ''} "
          f"— {plan['plan_tipo']}")
    print("=" * 70)
    print(generar_explicacion(plan))
    print()

### 8b. Glosario y botón «Actualizar mi día»

Dos piezas de la aplicación. El **glosario** alimenta las ayudas (icono ⓘ) junto a cada métrica y cada término
del plan: un deportista que empieza no tiene por qué saber qué es el TSB o un brick. El botón **«Actualizar
mi día»** recalcula solo la sesión de hoy con las métricas de esa mañana (mala noche, enfermedad, estrés alto)
sin rehacer la semana. Hay tres respuestas: mantener, suavizar (60 % de la duración y todo en zonas 1 y 2) o
descansar, y la sesión que se pierde se reprograma (memoria, 4.5.4).

In [ ]:
# --- Glosario: alimenta las ayudas (ⓘ) de la aplicación ---
GLOSARIO = {
    # Métricas fisiológicas
    'Body Battery': "Reserva de energía (0-100) que estima el reloj combinando HRV, estrés, sueño y actividad. Por debajo de 20-30, el cuerpo pide recuperación, no entrenamiento.",
    'HRV': "Variabilidad de la frecuencia cardiaca: los milisegundos que cambian entre latido y latido. Alta y estable = bien recuperado; si cae de golpe, fatiga, estrés o enfermedad.",
    'Estrés': "Índice 0-100 derivado de la HRV. Sostenido por encima de 70 el sistema nervioso está en alerta y asimila peor el entrenamiento.",
    'Sueño profundo': "Fase del sueño donde ocurre la reparación muscular y hormonal. Menos de ~1 h por noche de forma repetida frena la adaptación.",
    'FC': "Frecuencia cardiaca, en pulsaciones por minuto (ppm). Es el cuentarrevoluciones del entrenamiento.",
    'FC máxima / reposo': "Tus dos extremos: la FC máxima observada en esfuerzos duros y la FC al despertar. Entre ambas se calculan tus zonas (método de Karvonen).",
    'CTL (fitness)': "Carga crónica: media de tu entrenamiento de ~6 semanas. Es tu forma física acumulada; sube despacio y se pierde despacio.",
    'ATL (fatiga)': "Carga aguda: media de ~7 días. Es el cansancio reciente; sube y baja rápido.",
    'TSB (frescura)': "Fitness menos fatiga. Negativo = sobrecarga (entrenando fuerte); positivo = fresco. Muy negativo (<-25) = riesgo; muy positivo en carrera = llegas a punto.",
    'TRIMP': "Impulso de entrenamiento: cuánto 'pesa' una sesión combinando duración e intensidad cardiaca.",
    # Términos del plan
    'Z1-Z5 (zonas)': "Intensidades por FC: Z1 muy suave (recuperar), Z2 aeróbico (la base, puedes hablar), Z3 medio (te cuesta hablar), Z4 umbral (frases cortas), Z5 máximo (no hablas).",
    'Z2': "La zona donde se construye la resistencia del 70.3. Parece 'demasiado fácil': esa es la señal de que lo estás haciendo bien.",
    'Z4 / umbral': "El ritmo más alto que aguantas ~1 hora. Las series aquí elevan tu techo, pero solo cuando la base ya existe.",
    'D+': "Desnivel positivo acumulado: metros totales de subida de la ruta. Más D+ = más carga muscular y cardiaca al mismo ritmo.",
    'brick': "Bici seguida inmediatamente de carrera. Entrena la transición T2: los primeros minutos con 'piernas de cemento' que sentirás el día del 70.3.",
    'series': "Repeticiones de un esfuerzo con recuperación entre ellas (ej. 6x800 m). Permiten acumular minutos a intensidades que de seguido no aguantarías.",
    'fartlek': "Juego de ritmos dentro de un rodaje: cambios cortos más rápidos sin parar. Intensidad controlada para fases tempranas.",
    'FTP': "Potencia máxima sostenible ~1 h en bici (vatios). Con potenciómetro, las series se hacen al 90-100% del FTP; sin él, por FC y velocidad.",
    'cadencia': "Pedaladas (rpm) o zancadas por minuto. En bici, 85-95 rpm protege las piernas para la carrera posterior; en cuestas, 70-75.",
    'taper': "Reducción del volumen (~50%) las 2-3 semanas previas a la carrera manteniendo toques de intensidad: la fatiga se va, la forma se queda.",
    'pacing': "Estrategia de ritmo de carrera: a qué velocidad/FC afrontar cada segmento del 70.3 para no fundirte antes de la media maratón.",
}

def ayuda(termino):
    """Búsqueda flexible en el glosario (en la aplicación, una ayuda por término)."""
    t = termino.lower()
    for k, v in GLOSARIO.items():
        if t in k.lower(): return f"{k}: {v}"
    return f"'{termino}' no está en el glosario."

# --- Botón «Actualizar mi día»: recalcula SOLO la sesión de hoy ---
DIAS_ES = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']

def actualizar_hoy(plan, sueno_h=None, body_battery=None, stress=None,
                   hrv=None, enfermo=False):
    """Recalcula la recomendación del día con las métricas de esta mañana,
    sin rehacer la semana. Tres niveles: MANTENER, SUAVIZAR (−40% de
    duración, todo Z1-Z2, sin calidad) o DESCANSO (la sesión se reprograma,
    no se pierde). En la aplicación es el botón «Actualizar mi día»."""
    dia = DIAS_ES[pd.Timestamp.today().dayofweek]
    sesiones_hoy = [s for s in plan['sesiones'] if s['dia'].split(' ')[0] == dia]
    z = plan['zonas']

    graves, moderados = [], []
    if enfermo:
        graves.append('enfermedad o malestar')
    if body_battery is not None and body_battery < 20:
        graves.append(f'Body Battery muy baja ({body_battery:.0f})')
    if sueno_h is not None and sueno_h < 5:
        graves.append(f'noche muy corta ({sueno_h:.1f} h)')
    if not graves:
        if body_battery is not None and body_battery < 35:
            moderados.append(f'Body Battery baja ({body_battery:.0f})')
        if sueno_h is not None and sueno_h < 6.5:
            moderados.append(f'sueño insuficiente ({sueno_h:.1f} h)')
        if stress is not None and stress > 70:
            moderados.append(f'estrés alto ({stress:.0f})')
        if (hrv is not None and not pd.isna(plan.get('hrv', np.nan))
                and hrv < plan['hrv'] * 0.85):
            moderados.append(f'HRV un 15% por debajo de tu media ({hrv:.0f} ms)')

    if graves:
        ajuste = {'decision': 'DESCANSO HOY',
                  'sesion_ajustada': 'Descanso completo: caminar suave, hidratarte y priorizar dormir.',
                  'nota': ('La sesión prevista no se pierde: se reprograma y la semana se '
                           'recompone sin acumular carga extra. Si era de calidad, se '
                           'intercambia con un rodaje.')}
    elif moderados:
        partes = []
        for s in sesiones_hoy:
            if s['deporte'] in ('Descanso', 'Movilidad'): continue
            dur = int(s['dur_min'] * 0.6)
            partes.append(f"{s['deporte']}: versión suave de {dur} min, todo en Z1-Z2 "
                          f"({z['hr']['Z1'][0]}-{z['hr']['Z2'][1]} ppm), sin series, "
                          f"ritmos ni rectas.")
        ajuste = {'decision': 'SUAVIZAR HOY',
                  'sesion_ajustada': ' '.join(partes) if partes else 'Hoy ya era descanso: mantenlo.',
                  'nota': ('Día regenerativo puntual: mañana se reevalúa. Si hoy tocaba '
                           'calidad, se intercambia con el rodaje de otro día de la semana.')}
    else:
        ajuste = {'decision': 'MANTENER',
                  'sesion_ajustada': ('; '.join(s['detalle'] for s in sesiones_hoy)
                                      if sesiones_hoy else 'Día de descanso programado.'),
                  'nota': 'Tus métricas de esta mañana están en tu rango: el plan sigue tal cual.'}
    ajuste['dia'] = dia
    ajuste['motivos'] = graves + moderados or ['métricas en rango']
    return ajuste

# Ejemplo del botón: el deportista A amanece tras una mala noche
plan_a = next((p for p in planes if p['athlete'] == SUJETO_PRUEBA), planes[0])
demo = actualizar_hoy(plan_a, sueno_h=4.9, body_battery=28, stress=62)
print(f"«Actualizar mi día» — {demo['dia']} | decisión: {demo['decision']}")
print(f"  Motivos: {', '.join(demo['motivos'])}")
print(f"  Sesión ajustada: {demo['sesion_ajustada']}")
print(f"  Nota: {demo['nota']}")
print()
print("Glosario:", len(GLOSARIO), "términos. Ejemplo ->", ayuda('brick'))

# --- Cuadro final «Para entender tu plan» + ilustración de zonas ---
TERMINOS_DEL_PLAN = ['FC', 'Z1-Z5 (zonas)', 'Z2', 'D+', 'brick', 'cadencia', 'TSB (frescura)']

def bloque_glosario(plan=None):
    """Cuadro explicativo que cierra cada recomendación. En la aplicación es el
    desplegable «Para entender tu plan», con el gráfico de zonas."""
    out = ['─' * 64, 'PARA ENTENDER TU PLAN']
    for k in TERMINOS_DEL_PLAN:
        out.append(f"  · {k} — {GLOSARIO[k]}")
    return '\n'.join(out)

GARMIN_COLORS = {'Z1': '#9aa0a6', 'Z2': '#3b82f6', 'Z3': '#22a06b',
                 'Z4': '#f59e0b', 'Z5': '#ef4444'}

def grafico_zonas(z, titulo='Tus zonas de FC, como las muestra tu reloj'):
    """Barra de zonas con los colores del reloj Garmin (gris, azul, verde,
    naranja, rojo) y los ppm reales del atleta, para asociarlas de
    inmediato."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 1.9))
    for zona, (lo, hi) in z['hr'].items():
        ax.barh(0, hi - lo, left=lo, height=0.62, color=GARMIN_COLORS[zona])
        ax.text((lo + hi) / 2, 0, f"{zona}\n{lo}-{hi}", ha='center', va='center',
                color='white', fontsize=9, fontweight='bold')
    ax.set_xlim(z['hr']['Z1'][0] - 8, z['hr']['Z5'][1] + 8)
    ax.set_yticks([])
    ax.set_xlabel('pulsaciones por minuto (ppm)', fontsize=9)
    for s in ('top', 'right', 'left'):
        ax.spines[s].set_visible(False)
    ax.set_title(titulo, fontsize=11, loc='left')
    plt.tight_layout()
    return fig

print()
print(bloque_glosario())
import matplotlib.pyplot as plt
grafico_zonas(plan_a['zonas']); plt.show()

### 8c. Plan de competencia, km a km

La semana de competencia incluye un plan de carrera por tramos de
kilómetros para las tres disciplinas, construido con las zonas y ritmos
objetivo del deportista. La clave fisiológica es que se sale de la natación
con la FC disparada (estrés del agua, neopreno y el
paso de horizontal a vertical), de modo que la bici arranca **progresiva
al 85 %** y solo sube al ritmo objetivo cuando la FC vuelve a su sitio.
Cada tramo lleva velocidad/ritmo, tope de FC, cadencia y nutrición.

In [ ]:
def plan_competencia(plan):
    """Plan de carrera por tramos de km, con las zonas y ritmos objetivo
    del atleta. Devuelve (segmentos, resumen de tiempo estimado)."""
    z = plan['zonas']; hr = z['hr']
    v_b, v_r, v_n = z['bici_obj'], z['run_obj'], z['nat_obj']
    seg = []
    add = lambda d, t, txt: seg.append({'disciplina': d, 'tramo': t, 'indicacion': txt})

    # ---- NATACIÓN (1,9 km) ----
    add('NATACIÓN 1,9 km', '0-300 m',
        "Salida CONTROLADA al 90%: deja pasar al grupo rápido y busca agua libre. "
        "El objetivo es no disparar la FC en los primeros minutos: la carrera no se gana aquí.")
    add('NATACIÓN 1,9 km', '300-1.600 m',
        f"Ritmo de crucero a {_pace_swim(v_n)}. Busca pies de alguien un punto más rápido "
        f"(el drafting es legal y ahorra ~5% de energía). Respira cada 2-3 brazadas.")
    add('NATACIÓN 1,9 km', '1.600-1.900 m',
        "Aumenta el pataleo suavemente: redirige sangre a las piernas antes de ponerte "
        "de pie y visualiza la T1.")
    add('TRANSICIÓN T1', '—',
        f"Saldrás del agua con la FC en {hr['Z4'][0]}+ ppm aunque no hayas apretado: "
        f"estrés del agua, neopreno y pasar de horizontal a vertical. Camina rápido, no esprintes.")

    # ---- BICI (90 km): arranque progresivo hasta recuperar la FC ----
    add('BICI 90 km', 'km 0-5',
        f"Arranque PROGRESIVO al 85% ({v_b * 0.85:.0f} km/h), cadencia 90-95 rpm. Tu FC viene "
        f"alta de la natación: NO la persigas; espera a que baje de {hr['Z3'][0]} ppm. Bebe ya.")
    add('BICI 90 km', 'km 5-15',
        f"Sube gradualmente hacia el ritmo objetivo ({v_b:.0f} km/h) SOLO cuando la FC se asiente "
        f"en Z2-Z3 bajo ({hr['Z2'][0]}-{hr['Z3'][0]} ppm). Primera comida en el km 10-15.")
    add('BICI 90 km', 'km 15-70',
        f"Velocidad de crucero {v_b:.0f} km/h, FC tope {hr['Z3'][1]} ppm. Come 60-90 g de "
        f"carbohidrato/hora (cada 30-40 min). En subidas, sentado y sin pasar de {hr['Z3'][1]} ppm "
        f"aunque te adelanten: lo devolverás corriendo.")
    add('BICI 90 km', 'km 70-85',
        f"El tramo del juicio: si las piernas van bien, MANTÉN {v_b:.0f} km/h — no subas. "
        f"Última comida sólida antes del km 80.")
    add('BICI 90 km', 'km 85-90',
        f"Afloja al 95% ({v_b * 0.95:.0f} km/h) y sube la cadencia a 95-100 rpm: piernas sueltas "
        f"para la T2. Estira la espalda y termina el bidón.")
    add('TRANSICIÓN T2', '—',
        "La FC vuelve a subir al desmontar: es normal. Cambio rápido pero sin pánico; "
        "gorra, dorsal y a correr.")

    # ---- CARRERA (21,1 km) ----
    add('CARRERA 21,1 km', 'km 0-3',
        f"BRICK: piernas de cemento los primeros 2-3 km. Sal a {_pace_run(v_r * 0.92)} aunque te "
        f"sientas capaz de más: la FC ya viene en {hr['Z3'][0]}+ ppm y debe asentarse.")
    add('CARRERA 21,1 km', 'km 3-10',
        f"Ritmo objetivo {_pace_run(v_r)}, FC tope {hr['Z3'][1]} ppm. Agua o isotónico en cada "
        f"avituallamiento; gel en el km 7-8.")
    add('CARRERA 21,1 km', 'km 10-16',
        f"Mantén {_pace_run(v_r)}. Aquí se decide la carrera: si la FC sube a Z4 "
        f"({hr['Z4'][0]}+ ppm) sin haber subido el ritmo, afloja 10-15 s/km y reevalúa.")
    add('CARRERA 21,1 km', 'km 16-19',
        f"La parte dura. Permitido entrar en Z4 bajo ({hr['Z4'][0]}-"
        f"{int((hr['Z4'][0] + hr['Z4'][1]) / 2)} ppm) si las sensaciones acompañan.")
    add('CARRERA 21,1 km', 'km 19-21,1',
        "Vacía el tanque: todo lo que quede. Cruzar la meta de un 70.3 se recuerda toda la vida.")

    # ---- Tiempo estimado por segmento ----
    t_nat = 1.9 / v_n * 60
    t_bici = (5 / (v_b * 0.85) + 10 / (v_b * 0.95) + 70 / v_b + 5 / (v_b * 0.95)) * 60
    t_run = (3 / (v_r * 0.92) + 18.1 / v_r) * 60
    trans = plan['objetivo']['transiciones_min'] if plan.get('objetivo') else 8
    total = t_nat + trans + t_bici + t_run
    resumen = (f"Tiempo estimado: natación {_min2t(t_nat)} + T1/T2 {_min2t(trans)} + "
               f"bici {_min2t(t_bici)} + carrera {_min2t(t_run)} = {_min2t(total)}")
    if plan.get('objetivo'):
        resumen += f" (objetivo declarado: {plan['objetivo']['objetivo_total']})"
    return seg, resumen


def texto_plan_competencia(plan):
    seg, resumen = plan_competencia(plan)
    out, disc = [], None
    for s in seg:
        if s['disciplina'] != disc:
            disc = s['disciplina']
            out.append(f"\n  {disc}")
        out.append(f"    {s['tramo']:>12s} | {s['indicacion']}")
    out.append(f"\n  {resumen}")
    return '\n'.join(out)


# --- Ejemplo: B como deportista experimentado con carrera este domingo ---
# Los tiempos de la marca anterior son FICTICIOS: ningún deportista del estudio
# ha competido en un 70.3. Sirven solo para mostrar el plan orientado a marca.
PERFIL_ATLETA['B'] = 'experimentado'
OBJETIVOS_70_3['B'] = {'anterior': {'nat': '0:42', 'bici': '2:58',
                                    'run': '2:05', 'total': '5:55'},
                       'objetivo_total': '5:35'}
FECHA_COMPETENCIA['B'] = str((pd.Timestamp.today() + pd.Timedelta(days=6)).date())

if 'B' in garmin_data:
    plan_b = generar_plan_semanal('B')
    print("Ejemplo con tiempos de marca FICTICIOS (ningún deportista del estudio ha competido en un 70.3)")
    print()
    print(generar_explicacion(plan_b))
else:
    print("(B no está en los datos cargados)")

## 9. Almacenamiento de resultados (Parquet, SQLite y joblib)

Memoria, apartado 4.3 y Anexo B. Parquet guarda las tablas grandes (actividades, serie diaria de carga y
variables de cada deportista); SQLite guarda los resultados del cuaderno (planes, sesiones y predicciones);
joblib guarda los dos modelos de transferencia y las listas de variables que esperan, que son lo único que
necesita la aplicación.

In [ ]:
import sqlite3
import joblib

os.makedirs('salidas/parquet', exist_ok=True)
os.makedirs('salidas/modelos', exist_ok=True)

# 1) Parquet: datos voluminosos, comprimidos por columnas
if not df_act.empty:
    df_act.to_parquet('salidas/parquet/actividades.parquet', index=False, compression='snappy')
if fitness_data:
    pd.concat(fitness_data.values(), ignore_index=True).to_parquet(
        'salidas/parquet/fitness_fatiga.parquet', index=False, compression='snappy')
df_profiles.reset_index().to_parquet('salidas/parquet/perfiles.parquet', index=False)

# 2) SQLite: resultados del cuaderno (variables, planes, sesiones y predicciones)
con = sqlite3.connect('salidas/triatlon.db')
df_profiles.reset_index().to_sql('perfiles', con, if_exists='replace', index=False)
df_planes.to_sql('planes_semanales', con, if_exists='replace', index=False)
df_sesiones.to_sql('sesiones_plan', con, if_exists='replace', index=False)
pd.DataFrame(predicciones_cyc).to_sql('predicciones_ciclismo', con, if_exists='replace', index=False)
if predicciones_run:
    pd.DataFrame(predicciones_run).to_sql('predicciones_carrera', con, if_exists='replace', index=False)

# comprobación rápida de lo que quedó guardado
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)['name'].tolist()
print("Tablas en triatlon.db:", tablas)
con.close()

# 3) Modelos para la aplicación. La app lee de preprocesadores.joblib qué variables
#    espera cada modelo, así que no hay que tocar su código al cambiarlas.
joblib.dump(model_xgb,     'salidas/modelos/xgb_run2cyc.joblib')
joblib.dump(model_xgb_run, 'salidas/modelos/xgb_cyc2run.joblib')
joblib.dump({'feature_cols': FEATURE_COLS, 'feat_cyc2run': FEAT_CYC2RUN},
            'salidas/modelos/preprocesadores.joblib')
# Copia en el formato propio de XGBoost, independiente de la versión de joblib
model_xgb.save_model('salidas/modelos/xgb_run2cyc.json')
model_xgb_run.save_model('salidas/modelos/xgb_cyc2run.json')

# tamaño de cada fichero
for root, _, fnames in os.walk('salidas'):
    for fn in fnames:
        fp = os.path.join(root, fn)
        print(f"  {fp}: {os.path.getsize(fp) / 1024:.1f} KB")

print("\nResultados guardados en salidas/. La sección 11 los empaqueta en salidas.zip.")

## 10. Panel de control del cuaderno

Vista rápida para revisar los resultados durante el desarrollo. La herramienta para el deportista es la
aplicación de Streamlit (memoria, apartado 4.7).

In [ ]:
if fitness_data:
    fig = make_subplots(rows=2, cols=2,
        subplot_titles=['CTL/TSB (último año)', 'Actividades por deporte (horas)',
                        'FC media por participante', 'Velocidad ciclismo proyectada'],
        specs=[[{'type': 'scatter'}, {'type': 'bar'}], [{'type': 'box'}, {'type': 'bar'}]])

    colors_list = [COLORS.get(n, '#888') for n in fitness_data]

    for i, (name, fd) in enumerate(fitness_data.items()):
        fd_p = fd.tail(365); c = colors_list[i]
        fig.add_trace(go.Scatter(x=fd_p['date'], y=fd_p['CTL'], name=f'{name} CTL',
            line=dict(color=c, width=2), legendgroup=name, showlegend=True), row=1, col=1)
        fig.add_trace(go.Scatter(x=fd_p['date'], y=fd_p['TSB'], name=f'{name} TSB',
            line=dict(color=c, width=1, dash='dot'), legendgroup=name, showlegend=False), row=1, col=1)

    if not df_act.empty:
        sport_dur = df_act.groupby(['athlete', 'sport'])['duration_min'].sum().reset_index()
        for i, (name, grp) in enumerate(sport_dur.groupby('athlete')):
            fig.add_trace(go.Bar(x=grp['sport'], y=grp['duration_min'] / 60, name=name,
                marker_color=colors_list[i], legendgroup=name, showlegend=False), row=1, col=2)
        for i, (name, grp) in enumerate(df_act.groupby('athlete')):
            fig.add_trace(go.Box(y=grp['avg_hr'].dropna(), name=name,
                marker_color=colors_list[i], showlegend=False), row=2, col=1)

    # velocidades de ciclismo proyectadas (estimado conservador)
    if predicciones_cyc:
        df_pred = pd.DataFrame(predicciones_cyc)
        cols_bar = [COLORS.get(n, '#888') for n in df_pred['athlete']]
        fig.add_trace(go.Bar(x=df_pred['athlete'], y=df_pred['pred_conservador_kmh'],
            marker_color=cols_bar, text=df_pred['pred_conservador_kmh'].astype(str) + ' km/h',
            textposition='outside', showlegend=False), row=2, col=2)

    fig.update_layout(height=680, title_text='Panel de control — triatlón 70.3',
                      template='plotly_white', hovermode='x unified',
                      legend=dict(orientation='h', yanchor='bottom', y=1.02))
    fig.update_yaxes(title_text='TRIMP', row=1, col=1)
    fig.update_yaxes(title_text='Horas totales', row=1, col=2)
    fig.update_yaxes(title_text='FC media (bpm)', row=2, col=1)
    fig.update_yaxes(title_text='Velocidad km/h', row=2, col=2)
    fig.show()

print("\nCuaderno ejecutado completo. Estado final de cada deportista:")
for name, row in df_profiles.iterrows():
    nota = " (caso de evaluación)" if name == SUJETO_PRUEBA else ""
    print(f"  Atleta {name}: CTL={row.get('ctl_current', np.nan):.1f} | TSB={row.get('tsb_current', np.nan):.1f}{nota}")

## 11. Empaquetar `salidas/` para la aplicación

Crea `salidas.zip` y, en Colab, lo descarga. Para actualizar la aplicación basta con descomprimirlo junto a
`app.py` (o subir `salidas/modelos/` al repositorio) y volver a desplegar.

In [ ]:
import shutil

ESPERADOS = ['salidas/modelos/xgb_run2cyc.joblib', 'salidas/modelos/xgb_cyc2run.joblib',
             'salidas/modelos/xgb_run2cyc.json', 'salidas/modelos/xgb_cyc2run.json',
             'salidas/modelos/preprocesadores.joblib', 'salidas/triatlon.db',
             'salidas/parquet/actividades.parquet', 'salidas/parquet/fitness_fatiga.parquet',
             'salidas/parquet/perfiles.parquet']
faltan = [f for f in ESPERADOS if not os.path.exists(f)]
print("Faltan:", faltan if faltan else "nada, están todos los ficheros")

ruta_zip = shutil.make_archive('salidas', 'zip', '.', 'salidas')
print(f"Creado {ruta_zip} ({os.path.getsize(ruta_zip) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(ruta_zip)
except ImportError:
    pass